# Transition from [Part 1](https://colab.research.google.com/drive/14rMEfUlrJrWYFR42DYfBsLxMib1i2J6n?usp=sharing) → Part 2

Slide [RAG_Part_2](https://drive.google.com/file/d/1N1Y45c2gSu8nR0suw-JfDcoKO0Q3VvVe/view?usp=sharing)

In Part 1, we built a **Hybrid Compliance Decision Agent** that combines:

- Semantic Retrieval (in-memory embeddings)
- Deterministic Numeric Constraint Matching
- LLM-based Explanation Layer (Phi-3.5-mini-instruct)

The system successfully separated:

Retrieval  
Rule Validation  
LLM Reasoning  

This gave us:

- Reduced hallucination risk
- Deterministic business logic control
- Structured decision outputs

However, the architecture had important limitations.

---

#  Limitations in Part 1

Although the system was architecturally correct, it was not production-scalable:

- Embeddings were stored in memory
- No persistent vector database
- No ANN indexing
- No large-scale retrieval capability
- No proper benchmarking between retrieval vs decision correctness

This means:

It worked as a prototype —  
but not as a production-grade retrieval system.

---

# What Changes in Part 2?

In this file, we upgrade the system to a **real production-style retrieval architecture** by:

1️⃣ Replacing in-memory embeddings with a persistent Vector Database (Chroma)  
2️⃣ Using ANN indexing internally  
3️⃣ Supporting metadata filtering  
4️⃣ Benchmarking Vector-Only vs Hybrid systems  
5️⃣ Adding evaluation metrics  
6️⃣ Adding production risk handling  

---

# Core Insight of This Transition

Part 1 proved:
Hybrid Architecture works.

Part 2 proves:
Hybrid Architecture must be benchmarked and production-optimized.

We now answer a deeper question:

> Is retrieval quality enough for decision correctness?

Short answer:
No.

---

# 📌 Evolution Summary

| Layer | Part 1 | Part 2 |
|-------|--------|--------|
| Embeddings | In-memory | Persistent Vector DB |
| Retrieval | Cosine similarity | ANN + Metadata filter |
| Rule Engine | Yes | Yes (improved) |
| LLM Layer | Yes | Optional / explainable |
| Evaluation | Basic testing | Formal benchmarking |
| Risk Handling | Minimal | Production-aware |
| Scalability | Prototype | Production-oriented |

---

# Architectural Maturity

Part 1 → Hybrid Prototype  
Part 2 → Hybrid Production System  

Vector DB retrieves what is similar.  
Constraint engine selects what is correct.  
Evaluation proves the difference.

Now we move from:

"Does it work?"  

to  

"Is it safe, scalable, and benchmarked?"

---

---


# Hybrid Compliance Decision Engine  
### (RAG + Vector DB + Deterministic Constraint Engine)

This project implements a **production-style Hybrid Compliance System** that combines:

- Vector Database Retrieval (Chroma)
- Deterministic Numeric Constraint Engine
- Structured Decision Layer
- Evaluation & Benchmarking Framework
- Production Risk Handling
- Enterprise-Grade Extensions


# 🎯 Project Objective

To demonstrate why:

> Retrieval Quality ≠ Decision Quality

And how to build a **safe, explainable, auditable AI decision system** for compliance and financial policy automation.

---

# What This File Contains

## PART 1 — Data & Vector Database Setup
- Load policy CSV
- Clean and structure policy text
- Generate embeddings using SentenceTransformer
- Store policies in Chroma (Persistent Vector DB)
- Implement semantic search layer

## PART 2 — Hybrid Retrieval Architecture
- Demonstrate limitation of vector-only retrieval
- Show failure on numeric rules (e.g. 4000$ case)
- Implement numeric constraint matching
- Build hybrid policy selector:
    - Vector Retrieval  +  Deterministic Rule Validation =  Correct Policy Selection

## PART 3 — Production Decision Layer
- Build structured decision object
- Add numeric constraint validation trace
- Add explainable decision UI
- Ensure system is auditable and traceable

## PART 4 — Evaluation & Benchmarking
- Compare:
  - Vector-Only System
  - Hybrid System
- Metrics:
  - Hit Rate@K
  - Recall@K
  - Constraint Accuracy
  - Latency
- Generate 20 synthetic evaluation queries
- Prove hybrid architecture improves decision correctness

## PART 5 — Production Risk Handling
- Handle ambiguous monetary inputs (e.g., "4k")
- Add normalization layer
- Add ambiguity detection
- Add confidence-aware decision policy

## PART 6 — Enterprise System Extensions
- LLM-based section routing
- Confidence scoring layer
- Escalation mechanism
- ReAct-style agent transformation
- Tool-based architecture

#  Architectural Insight

Vector Database ≠ Rule Engine  
Vector Database ≠ Business Logic  

Vector DB retrieves what is *similar*.  
Constraint Engine selects what is *correct*.  
Confidence Engine decides whether it is *safe to act*.

#  Production Principles Demonstrated

- Hybrid AI Architecture
- Deterministic Safety Layer
- Explainable Decisions
- Auditability
- Confidence-Based Automation
- Risk-Aware System Design

# Key Learning Outcome

After completing this file, you should understand:

- Why semantic similarity alone is not enough
- How to combine AI with rule-based logic
- How to benchmark AI systems properly
- How to design enterprise-grade AI compliance systems
- How to safely move from pipeline → agent architecture


 This is not just a notebook.  
It is a blueprint for building safe, production-grade AI systems.

# PART 1 — Data & Vector DB Setup

In [1]:
!pip -q install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [2]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00


In [3]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


## First time

### NOTE: Chroma uses ANN indexing internally (not brute force search)

## ChromaDB (Quick Intro)

## What is ChromaDB?
**ChromaDB** is a lightweight **Vector Database** designed for **RAG** systems.

It helps you:
- store **embeddings** (vectors)
- search them fast using **ANN** (Approximate Nearest Neighbor)
- store **documents + metadata**
- apply **metadata filtering** (e.g., section = Financial)
- persist data on disk (so it stays after restart)

## The Core Idea (Simple)
Text → Embedding → Store in Chroma → Query → Get Top-K similar chunks

Chroma does NOT understand logic or numbers.
It only retrieves what is *semantically similar*.

## What Chroma Stores
A Chroma collection usually stores:
- `ids` (unique IDs)
- `documents` (text chunks)
- `embeddings` (vector for each chunk)
- `metadatas` (extra info like section, policy_name, tenant_id, etc.)

## What Happens When You Query
When you run a query:
1) Chroma converts your query into an embedding (you do this using an embedder)
2) Chroma searches using an **ANN index** (not brute force)
3) It returns the **Top-K closest vectors**
4) You can filter results using metadata (optional)

## Metadata Filtering (Production Feature)
Filtering means:
"Search only inside a subset of documents"

Example:
- `where = {"section": "Financial"}`
So Chroma searches only within *Financial* policies.

This is very important for:
- multi-tenant systems
- access control
- compliance rules
- scoped retrieval

## Important Limitation (Big Warning)
**Vector DB ≠ Rule Engine**

Chroma can retrieve:
✅ "Which rule sounds similar?"

But it cannot decide:
❌ "Which rule is logically correct based on amount > 3000?"

That’s why production systems use:
**Vector Retrieval + Constraint/Rule Engine** (Hybrid Architecture)

---

## When to Use ChromaDB
Use Chroma when you want:
- fast RAG prototype
- local vector storage
- simple production apps
- metadata filtering
- easy persistence


## When NOT to Use ChromaDB
Avoid Chroma if you need:
- massive scale (hundreds of millions/billions)
- distributed multi-node vector search
- full enterprise vector DB features

In that case, consider:
Milvus / Pinecone / Weaviate / Elasticsearch Vector

In [6]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer

import chromadb
from chromadb.config import Settings

def clean_text(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s))
    return s.strip()

# 1) Load policies
POLICY_CSV = "/content/Policy_en.csv"
df = pd.read_csv(POLICY_CSV)

cols = ["section","policy_name","rule_description","condition","approval_required","risk_level"]
df[cols] = df[cols].fillna("")

df["full_text"] = df.apply(lambda r: clean_text(
    f"{r['section']} | {r['policy_name']} | "
    f"{r['rule_description']} | {r['condition']} | "
    f"{r['approval_required']} | {r['risk_level']}"
), axis=1)

texts = df["full_text"].tolist()

# 2) Embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
PERSIST_DIR = "/content/drive/MyDrive/hf_models/agent_project/chroma_policy_db"

In [8]:
# 3) Create Persistent Chroma client

client = chromadb.PersistentClient(
    path=PERSIST_DIR,
    settings=Settings(anonymized_telemetry=False)
)

# 4) Create / get collection
collection = client.get_or_create_collection(name="policies_v1")

# 5) Prepare ids + metadatas
ids = [f"policy_{i}" for i in range(len(df))]

metadatas = []
for i, row in df.iterrows():
    metadatas.append({
        "section": str(row["section"]),
        "policy_name": str(row["policy_name"]),
        "approval_required": str(row["approval_required"]),
        "risk_level": str(row["risk_level"]),
        "condition": str(row["condition"]),
    })

# 6) Add to Vector DB
# (مهم: embeddings لازم تكون list of lists)
collection.add(
    ids=ids, # new: ids
    documents=texts, # raw text
    embeddings=embeddings.tolist(),# raw embedding
    metadatas=metadatas # new: metadatas
)

print("✅ Inserted into Chroma:", collection.count())
print("📁 Persisted at:", PERSIST_DIR)

✅ Inserted into Chroma: 26
📁 Persisted at: /content/drive/MyDrive/hf_models/agent_project/chroma_policy_db


## 2th time

In [9]:
import chromadb
from chromadb.config import Settings

client2 = chromadb.PersistentClient(
    path=PERSIST_DIR,
    settings=Settings(anonymized_telemetry=False)
)

collection2 = client2.get_collection(name="policies_v1")
print("✅ Loaded collection count:", collection2.count())

✅ Loaded collection count: 26


In [10]:
collection  = collection2

In [11]:
from sentence_transformers import SentenceTransformer
# Embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def chroma_search(query: str, k: int = 5, where: dict | None = None):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    res = collection.query( # from chroma
        query_embeddings=q_emb.tolist(),
        n_results=k,
        where=where  # metadata filtering (optional)
    )
    # res keys: ids, documents, metadatas, distances
    out = []
    for i in range(len(res["ids"][0])):
        out.append({
            "id": res["ids"][0][i],
            "distance": float(res["distances"][0][i]),
            "text": res["documents"][0][i],
            "metadata": res["metadatas"][0][i],
        })
    return out


In [13]:
query = "Purchase device with price 4000$ who approves?"

hits = chroma_search(query, k=5)
hits[:5]

[{'id': 'policy_1',
  'distance': 0.7570042610168457,
  'text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 1500 < amount <= 3000 | Manager | Medium',
  'metadata': {'approval_required': 'Manager',
   'risk_level': 'Medium',
   'section': 'Financial',
   'condition': '1500 < amount <= 3000',
   'policy_name': 'Hardware Purchase Policy'}},
 {'id': 'policy_0',
  'distance': 0.83439701795578,
  'text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | amount <= 1500 | No | Medium',
  'metadata': {'condition': 'amount <= 1500',
   'approval_required': 'No',
   'policy_name': 'Hardware Purchase Policy',
   'risk_level': 'Medium',
   'section': 'Financial'}},
 {'id': 'policy_2',
  'distance': 0.9324245452880859,
  'text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | amount > 3000 | Manager+Finance+CTO | Medium',
  'metadata': {'approv

In [14]:
len(hits)

5

# PART 2 — Hybrid Retrieval Layer - Rule_based

## Problem: Why Vector Database Alone Is Not Enough

After upgrading our system to a real Vector Database (Chroma), we tested the query:

> "Purchase device with price 4000$ who approves?"

However, the system returned the policy:

```
1500 < amount <= 3000
```

Instead of the correct rule:

```
amount > 3000
```
---

## Why Did This Happen?

Vector Databases use **semantic similarity**, not numeric reasoning.

The embedding model understands that both rules are related to:

- Hardware purchases
- Financial thresholds
- Approval rules

So linguistically, they are very similar.

But mathematically, they are very different.

The Vector DB does **not understand inequalities** like:

- `<`
- `>`
- `<=`
- `>=`

It only understands semantic closeness in embedding space.

---

## The Core Limitation

Vector DB can answer:

> "Which rule sounds most similar?"

But it cannot answer:

> "Which rule is logically correct based on the number 4000?"

This is a critical distinction.

If we rely only on semantic retrieval in financial or compliance systems, we risk:

- Incorrect approvals
- Wrong compliance decisions
- Legal exposure
- Business risk

---

# Architectural Insight

Vector Database ≠ Rule Engine  
Vector Database ≠ Business Logic  
Vector Database ≠ Deterministic System  

It is only a **retrieval layer**.

---

#  The Correct Solution: Hybrid Architecture

To build a reliable compliance system, we must combine:


Semantic Retrieval
↓
Deterministic Constraint Matching
↓
Final Policy Selection
↓
LLM Explanation


### Step 1 — Use Vector DB to retrieve top-k candidates  
### Step 2 — Apply numeric constraint matching in Python  
### Step 3 — Select the logically correct rule  
### Step 4 — Pass the validated rule to the LLM for explanation  

---

# Final Takeaway

Vector DB retrieves what is *similar*.  
Rule Engine selects what is *correct*.  

A production-grade system must use both.


In [15]:
import re

def extract_numbers(text: str):
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    return [float(n) for n in nums]

In [16]:
def match_numeric_condition(cond: str, value: float) -> bool:
    cond = cond.strip()

    # Case 1: 1500 < amount <= 3000
    m = re.match(r"^\s*([0-9\.]+)\s*<\s*\w+\s*<=\s*([0-9\.]+)\s*$", cond)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        return lo < value <= hi

    # Case 2: amount > 3000
    m = re.match(r"^\s*\w+\s*(<=|>=|<|>|==)\s*([0-9\.]+)\s*$", cond)
    if m:
        op, rhs = m.group(1), float(m.group(2))
        if op == "<=": return value <= rhs
        if op == ">=": return value >= rhs
        if op == "<":  return value < rhs
        if op == ">":  return value > rhs
        if op == "==": return value == rhs

    return False

In [17]:
def select_best_policy(query: str, hits: list):
    nums = extract_numbers(query)

    # 1️⃣ Try numeric matching first
    if nums:
        for h in hits:
            cond = h["metadata"].get("condition", "")
            for n in nums:
                if match_numeric_condition(cond, n):
                    return h, True  # constraint used - Rule_based

    # 2️⃣ Fallback → highest similarity
    return hits[0] if hits else None, False

In [18]:
def hybrid_policy_retrieval(query: str, k: int = 5, where: dict | None = None):
    """
    Hybrid retrieval:
    1) Vector similarity search (with optional metadata filtering)
    2) Deterministic numeric constraint matching
    """

    # 1️⃣ Retrieve from Vector DB (with metadata filter if provided)
    hits = chroma_search(query, k=k, where=where)

    if not hits:
        return {
            "best_match": None,
            "constraint_used": False,
            "candidates": [],
            "filter_used": where
        }

    # 2️⃣ Apply numeric constraint matching
    best, constraint_used = select_best_policy(query, hits) # rule_Based

    return {
        "best_match": best,
        "constraint_used": constraint_used,
        "candidates": hits,
        "filter_used": where
    }

In [19]:
query = "Purchase device with price 4000$ who approves?"

result = hybrid_policy_retrieval(query, k=5)

print("Constraint Used:", result["constraint_used"])
print("Selected Policy:\n", result["best_match"])

Constraint Used: True
Selected Policy:
 {'id': 'policy_2', 'distance': 0.9324245452880859, 'text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | amount > 3000 | Manager+Finance+CTO | Medium', 'metadata': {'section': 'Financial', 'risk_level': 'Medium', 'condition': 'amount > 3000', 'policy_name': 'Hardware Purchase Policy', 'approval_required': 'Manager+Finance+CTO'}}


In [20]:
result["best_match"]

{'id': 'policy_2',
 'distance': 0.9324245452880859,
 'text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | amount > 3000 | Manager+Finance+CTO | Medium',
 'metadata': {'section': 'Financial',
  'risk_level': 'Medium',
  'condition': 'amount > 3000',
  'policy_name': 'Hardware Purchase Policy',
  'approval_required': 'Manager+Finance+CTO'}}

In [ ]:
query = "who policy need Manager+Finance approves?"

result = hybrid_policy_retrieval(
    query,
    k=5,
    where={"section": "Security"}  # 🔥 metadata filter
)

print("Constraint Used:", result["constraint_used"])
result["best_match"]

Constraint Used: False


{'id': 'policy_15',
 'distance': 1.3162676095962524,
 'text': 'Security | Data Handling Policy | External sharing requires legal approval | external_sharing == true | Legal | Critical',
 'metadata': {'policy_name': 'Data Handling Policy',
  'approval_required': 'Legal',
  'condition': 'external_sharing == true',
  'risk_level': 'Critical',
  'section': 'Security'}}

In [22]:
result

{'best_match': {'id': 'policy_15',
  'distance': 1.3162676095962524,
  'text': 'Security | Data Handling Policy | External sharing requires legal approval | external_sharing == true | Legal | Critical',
  'metadata': {'policy_name': 'Data Handling Policy',
   'approval_required': 'Legal',
   'condition': 'external_sharing == true',
   'risk_level': 'Critical',
   'section': 'Security'}},
 'constraint_used': False,
 'candidates': [{'id': 'policy_15',
   'distance': 1.3162676095962524,
   'text': 'Security | Data Handling Policy | External sharing requires legal approval | external_sharing == true | Legal | Critical',
   'metadata': {'policy_name': 'Data Handling Policy',
    'approval_required': 'Legal',
    'condition': 'external_sharing == true',
    'risk_level': 'Critical',
    'section': 'Security'}},
  {'id': 'policy_11',
   'distance': 1.339461088180542,
   'text': 'Security | Software Installation Policy | SaaS handling customer data requires security review and DPA | handles_cu

# PART 3 — Production Decision Layer

## Production = Explainable + Auditable + Observable
##  1 — Build Decision Object
##  2 —Add Constraint Validation Trace

In [23]:
def extract_numeric_evidence(query: str, best_match: dict):
    nums = extract_numbers(query)
    cond = best_match["metadata"].get("condition", "")

    evidence = []

    for n in nums:
        match = match_numeric_condition(cond, n)
        evidence.append({
            "query_value": n,
            "condition": cond,
            "matched": match
        })

    return evidence

In [62]:
def build_decision_output(result: dict, query: str):
    best = result["best_match"]

    if not best:
        return {
            "decision": None,
            "error": "No policy matched",
            "timestamp": datetime.datetime.utcnow().isoformat()
        }

    constraint_evidence = extract_numeric_evidence(query, best)

    decision = {
        "approval_required": best["metadata"].get("approval_required"),
        "risk_level": best["metadata"].get("risk_level")
    }

    return {
        "decision": decision,
        "policy": best,
        "constraint_used": result["constraint_used"],
        "constraint_evidence": constraint_evidence,
        "timestamp": datetime.datetime.utcnow().isoformat()
    }

In [43]:
query = "Purchase device with price 4000$ who approves?"
raw_result = hybrid_policy_retrieval(query, k=5)

prod_output = build_decision_output(raw_result, query)
prod_output

/tmp/ipykernel_5184/3664448687.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()


{'decision': {'approval_required': 'Manager+Finance+CTO',
  'risk_level': 'Medium'},
 'policy': {'id': 'policy_2',
  'distance': 0.9324245452880859,
  'text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | amount > 3000 | Manager+Finance+CTO | Medium',
  'metadata': {'approval_required': 'Manager+Finance+CTO',
   'policy_name': 'Hardware Purchase Policy',
   'section': 'Financial',
   'condition': 'amount > 3000',
   'risk_level': 'Medium'}},
 'constraint_used': True,
 'constraint_evidence': [{'query_value': 4000.0,
   'condition': 'amount > 3000',
   'matched': True}],
 'timestamp': '2026-03-29T12:23:51.448458'}

## 2 — Add Constraint Validation Trace

In [44]:
# @title
from IPython.display import display, HTML

def display_decision_ui(result: dict):
    decision = result["decision"]
    policy = result["policy"]
    metadata = policy["metadata"]
    evidence = result["constraint_evidence"][0]

    html = f"""
    <div style="border:2px solid #4CAF50; padding:15px; border-radius:10px;">
        <h2>✅ Compliance Decision</h2>

        <p><b>Approval Required:</b> {decision['approval_required']}</p>
        <p><b>Risk Level:</b> {decision['risk_level']}</p>

        <hr>
        <h3>📜 Policy Details</h3>
        <p><b>Section:</b> {metadata['section']}</p>
        <p><b>Policy Name:</b> {metadata['policy_name']}</p>
        <p><b>Full Policy Text:</b></p>
        <div style="background:#f4f4f4; padding:10px; border-radius:5px;">
            {policy['text']}
        </div>

        <hr>
        <h3>📏 Constraint Validation</h3>
        <p><b>Condition:</b> {evidence['condition']}</p>
        <p><b>Query Value:</b> {evidence['query_value']}</p>
        <p><b>Matched:</b> {evidence['matched']}</p>

        <hr>
        <small>🕒 {result['timestamp']}</small>
    </div>
    """

    display(HTML(html))

In [45]:
display_decision_ui(prod_output)

# PART 4 — Evaluation & Benchmarking


This section explains the evaluation metrics used to compare the **Vector-Only system** and the **Hybrid (Vector + Constraint Engine) system**.

---

## 1️⃣ Total Queries

**Definition:**  
The total number of test queries used during evaluation.

**Why it matters:**  
It defines the sample size of the experiment.  
A larger number increases reliability of the benchmark.

**Formula:**
```
Total Queries = len(evaluation_dataset)
```

---

## 2️⃣ Hit Rate@K

**Definition:**  
Measures whether the system returned *any result* in the Top-K retrieved candidates.

**Interpretation:**
- `1.0` → The system always retrieved results.
- `< 1.0` → Some queries returned no candidates.

**Formula:**
```
Hit Rate@K = Number of queries with at least one result / Total Queries
```

**Important:**  
Hit Rate does NOT measure correctness.  
It only measures retrieval coverage.

---

## 3️⃣ Recall@K

**Definition:**  
Checks whether the correct policy condition exists inside the Top-K retrieved candidates.

**Interpretation:**
- `1.0` → The correct rule was always retrieved.
- `< 1.0` → The correct rule was missing in some cases.

**Formula:**
```
Recall@K = Queries where correct rule is in Top-K / Total Queries
```

**Important:**  
Recall measures retrieval quality —  
NOT whether the system selected the correct rule.

---

## 4️⃣ Constraint Accuracy

**Definition:**  
Measures whether the final selected rule is logically correct.

**Interpretation:**
- `1.0` → Every final decision was correct.
- `0.35` → Only 35% of decisions were correct.

**Formula:**
```
Constraint Accuracy = Correct final decisions / Total Queries
```
This is the **true decision quality metric**.

It answers:
> Did the system select the correct policy?
---

## 5️⃣ Average Latency (ms)

**Definition:**  
The average time (in milliseconds) required to process one query.

**Formula:**
```
Average Latency = Total processing time / Total Queries
```

**Why it matters:**
- Lower latency = Faster system
- Important for production environments

---

| Metric | Measures |
|--------|----------|
| Hit Rate@K | Retrieval coverage |
| Recall@K | Retrieval completeness |
| Constraint Accuracy | Decision correctness |
| Latency | System efficiency |

---
High Recall does NOT guarantee correct decisions.  
Vector Search retrieves candidates.  
Constraint Engine ensures correctness.



In [46]:
evaluation_queries = [
    {
        "query": "Purchase device with price 4000$ who approves?",
        "expected_condition": "amount > 3000"
    },
    {
        "query": "Purchase device with price 2000$ who approves?",
        "expected_condition": "1500 < amount <= 3000"
    }
]

In [47]:
import time

def timed_hybrid(query, k=5):
    start = time.time()
    result = hybrid_policy_retrieval(query, k=k)
    end = time.time()

    latency_ms = (end - start) * 1000

    return result, latency_ms

In [48]:
def recall_at_k(result, expected_condition):
    """
    Checks if the correct condition exists
    inside top-k retrieved candidates.
    """
    for h in result["candidates"]:
        if h["metadata"]["condition"] == expected_condition:
            return 1
    return 0

In [49]:
def evaluate_system(eval_data, k=5):
    total = len(eval_data)
    hits = 0
    constraint_correct = 0
    recall_hits = 0
    total_latency = 0

    for item in eval_data:
        query = item["query"]
        expected = item["expected_condition"]

        result, latency = timed_hybrid(query, k=k)
        total_latency += latency

        best = result["best_match"]

        # Hit Rate
        if best:
            hits += 1

            predicted_condition = best["metadata"]["condition"]

            # Constraint Accuracy
            if predicted_condition == expected:
                constraint_correct += 1

        # Recall@K
        recall_hits += recall_at_k(result, expected)

    return {
        "Total Queries": total,
        "Hit Rate@K": hits / total,
        "Recall@K": recall_hits / total,
        "Constraint Accuracy": constraint_correct / total,
        "Average Latency (ms)": total_latency / total
    }

In [50]:
metrics = evaluate_system(evaluation_queries, k=5)
metrics

{'Total Queries': 2,
 'Hit Rate@K': 1.0,
 'Recall@K': 1.0,
 'Constraint Accuracy': 1.0,
 'Average Latency (ms)': 35.48288345336914}

In [51]:
def select_best_policy_vector_only(query: str, hits: list):
    """
    Vector-only selection:
    Always pick highest similarity result.
    No numeric reasoning.
    """
    return hits[0] if hits else None, False


def hybrid_policy_retrieval_vector_only(query: str, k: int = 5, where: dict | None = None):

    hits = chroma_search(query, k=k, where=where)

    if not hits:
        return {
            "best_match": None,
            "constraint_used": False,
            "candidates": [],
            "filter_used": where
        }

    best, constraint_used = select_best_policy_vector_only(query, hits)

    return {
        "best_match": best,
        "constraint_used": constraint_used,
        "candidates": hits,
        "filter_used": where
    }
def timed_vector_only(query, k=5):
    import time
    start = time.time()
    result = hybrid_policy_retrieval_vector_only(query, k=k)
    end = time.time()
    latency_ms = (end - start) * 1000
    return result, latency_ms
def evaluate_vector_only(eval_data, k=5):
    total = len(eval_data)
    hits = 0
    constraint_correct = 0
    recall_hits = 0
    total_latency = 0

    for item in eval_data:
        query = item["query"]
        expected = item["expected_condition"]

        result, latency = timed_vector_only(query, k=k)
        total_latency += latency

        best = result["best_match"]

        # Hit Rate
        if best:
            hits += 1

            predicted_condition = best["metadata"]["condition"]

            # Constraint Accuracy
            if predicted_condition == expected:
                constraint_correct += 1

        # Recall@K
        recall_hits += recall_at_k(result, expected)

    return {
        "Total Queries": total,
        "Hit Rate@K": hits / total,
        "Recall@K": recall_hits / total,
        "Constraint Accuracy": constraint_correct / total,
        "Average Latency (ms)": total_latency / total
    }


In [52]:
vector_metrics = evaluate_vector_only(evaluation_queries, k=5)
hybrid_metrics = evaluate_system(evaluation_queries, k=5)

print("Vector Only:", vector_metrics)
print("Hybrid:", hybrid_metrics)

Vector Only: {'Total Queries': 2, 'Hit Rate@K': 1.0, 'Recall@K': 1.0, 'Constraint Accuracy': 0.5, 'Average Latency (ms)': 26.951193809509277}
Hybrid: {'Total Queries': 2, 'Hit Rate@K': 1.0, 'Recall@K': 1.0, 'Constraint Accuracy': 1.0, 'Average Latency (ms)': 23.743748664855957}


Retrieval Quality ≠ Decision Quality
- Vector Search gives candidates.
- Constraint Engine gives correctness.

## Generate 20 Synthetic Queries

In [54]:
import random

def generate_synthetic_queries():
    queries = []

    test_values = [
        500, 1200, 1500, 1800, 2500,
        2999, 3000, 3001, 3500, 4000,
        4500, 5000, 750, 1600, 2700,
        3100, 10000, 200, 2200, 3300
    ]

    for value in test_values:
        if value <= 1500:
            expected = "amount <= 1500"
        elif 1500 < value <= 3000:
            expected = "1500 < amount <= 3000"
        else:
            expected = "amount > 3000"

        queries.append({
            "query": f"Purchase device with price {value}$ who approves?",
            "expected_condition": expected
        })

    return queries


evaluation_queries_20 = generate_synthetic_queries()
len(evaluation_queries_20)

20

In [55]:
evaluation_queries_20

[{'query': 'Purchase device with price 500$ who approves?',
  'expected_condition': 'amount <= 1500'},
 {'query': 'Purchase device with price 1200$ who approves?',
  'expected_condition': 'amount <= 1500'},
 {'query': 'Purchase device with price 1500$ who approves?',
  'expected_condition': 'amount <= 1500'},
 {'query': 'Purchase device with price 1800$ who approves?',
  'expected_condition': '1500 < amount <= 3000'},
 {'query': 'Purchase device with price 2500$ who approves?',
  'expected_condition': '1500 < amount <= 3000'},
 {'query': 'Purchase device with price 2999$ who approves?',
  'expected_condition': '1500 < amount <= 3000'},
 {'query': 'Purchase device with price 3000$ who approves?',
  'expected_condition': '1500 < amount <= 3000'},
 {'query': 'Purchase device with price 3001$ who approves?',
  'expected_condition': 'amount > 3000'},
 {'query': 'Purchase device with price 3500$ who approves?',
  'expected_condition': 'amount > 3000'},
 {'query': 'Purchase device with price 

In [56]:
vector_metrics_20 = evaluate_vector_only(evaluation_queries_20, k=5)
hybrid_metrics_20 = evaluate_system(evaluation_queries_20, k=5)

print("Vector Only Metrics:")
print(vector_metrics_20)

print("\nHybrid Metrics:")
print(hybrid_metrics_20)

Vector Only Metrics:
{'Total Queries': 20, 'Hit Rate@K': 1.0, 'Recall@K': 1.0, 'Constraint Accuracy': 0.35, 'Average Latency (ms)': 24.68775510787964}

Hybrid Metrics:
{'Total Queries': 20, 'Hit Rate@K': 1.0, 'Recall@K': 1.0, 'Constraint Accuracy': 1.0, 'Average Latency (ms)': 22.968482971191406}


# PART 5 — Ambiguity & Production Handling

In [63]:
import datetime

query = "Who signs off on a 4k hardware purchase?"

result = hybrid_policy_retrieval(
    query,
    k=5,
    where={"section": "Financial"}  # 🔥 metadata filter
)
prod_output = build_decision_output(result, query)
prod_output

/tmp/ipykernel_5184/4206085607.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow().isoformat()


{'decision': {'approval_required': 'No', 'risk_level': 'Medium'},
 'policy': {'id': 'policy_0',
  'distance': 0.9787430167198181,
  'text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | amount <= 1500 | No | Medium',
  'metadata': {'policy_name': 'Hardware Purchase Policy',
   'approval_required': 'No',
   'condition': 'amount <= 1500',
   'risk_level': 'Medium',
   'section': 'Financial'}},
 'constraint_used': True,
 'constraint_evidence': [{'query_value': 4.0,
   'condition': 'amount <= 1500',
   'matched': True}],
 'timestamp': '2026-03-29T12:28:35.423383'}

## Production Handling for Ambiguous Monetary Inputs (e.g., "4k")

## Problem

User query:
> "Who signs off on a 4k hardware purchase?"

If the system parses "4k" as `4` instead of `4000`,
it may produce a logically incorrect decision.

This is not a retrieval problem.
This is a **normalization + confidence problem**.

---

# Production-Grade Solution

## 1️⃣ Robust Normalization Layer

Normalize common monetary formats before extraction:

- 4k → 4000
- $4,000 → 4000
- 4 thousand → 4000
- USD 4k → 4000

Always normalize before numeric parsing.

---

## 2️⃣ Ambiguity Detection

If parsing is uncertain, do NOT auto-decide.

Example signals:
- Extracted value is suspiciously small
- Original query contains "k" but parsed value is low
- Multiple conflicting numbers detected

When ambiguity is detected:
→ Reduce confidence score
→ Trigger clarification or escalation

---

## 3️⃣ Confidence Scoring Layer

Every decision must include a confidence score.

Example components:
- Vector similarity score
- Constraint match (True/False)
- Section classification confidence
- Number parsing confidence

If confidence < threshold:
→ Do not auto-approve

---

## 4️⃣ Decision Policy

| Confidence Level | Action |
|------------------|--------|
| High             | Auto decision |
| Medium           | Ask clarification |
| Low              | Escalate to human |

---

# Core Principle

Vector DB retrieves what is similar.  
Constraint engine selects what is logically correct.  
Confidence engine decides whether it is safe to act.

If uncertain → Never auto-decide.

 Production Risk Handling Layer

In [64]:
import re
from datetime import datetime, timezone

# ─────────────────────────────────────────────
# 1️⃣  Robust Normalization Layer
# ─────────────────────────────────────────────

def normalize_monetary_input(query: str) -> tuple[str, bool]:
    """
    Normalize ambiguous monetary formats before numeric extraction.
    Returns (normalized_query, was_normalized).

    Handles:
        4k        → 4000
        4.5k      → 4500
        $4,000    → 4000
        4 thousand → 4000
        USD 4k    → 4000
    """
    original = query
    normalized = query

    # Remove currency symbols and commas  →  $4,000  →  4000
    normalized = re.sub(r"\$|USD\s*|,", "", normalized, flags=re.IGNORECASE)

    # Replace  Xk / X.Yk  →  X*1000
    normalized = re.sub(
        r"(\d+(?:\.\d+)?)\s*k\b",
        lambda m: str(int(float(m.group(1)) * 1000)),
        normalized,
        flags=re.IGNORECASE
    )

    # Replace  X thousand  →  X*1000
    normalized = re.sub(
        r"(\d+(?:\.\d+)?)\s*thousand\b",
        lambda m: str(int(float(m.group(1)) * 1000)),
        normalized,
        flags=re.IGNORECASE
    )

    was_normalized = normalized.strip() != original.strip()
    return normalized, was_normalized


# ─────────────────────────────────────────────
# 2️⃣  Ambiguity Detection
# ─────────────────────────────────────────────

def detect_ambiguity(original_query: str, normalized_query: str, was_normalized: bool) -> dict:
    """
    Detect signals that numeric parsing may be uncertain.

    Signals checked:
        - Query was normalized  (e.g. '4k' found)
        - Extracted value is suspiciously small  (< 10 after normalization)
        - Multiple conflicting numbers detected
    """
    raw_nums    = extract_numbers(original_query)
    norm_nums   = extract_numbers(normalized_query)

    suspicious_small   = any(n < 10 for n in norm_nums)
    multiple_numbers   = len(norm_nums) > 1
    k_pattern_present  = bool(re.search(r"\d+(?:\.\d+)?\s*k\b", original_query, re.IGNORECASE))

    is_ambiguous = was_normalized or suspicious_small or multiple_numbers

    return {
        "is_ambiguous"       : is_ambiguous,
        "was_normalized"     : was_normalized,
        "k_pattern_found"    : k_pattern_present,
        "suspicious_small"   : suspicious_small,
        "multiple_numbers"   : multiple_numbers,
        "original_numbers"   : raw_nums,
        "normalized_numbers" : norm_nums,
    }


# ─────────────────────────────────────────────
# 3️⃣  Confidence Scoring Layer
# ─────────────────────────────────────────────

CONFIDENCE_THRESHOLD_HIGH   = 0.75
CONFIDENCE_THRESHOLD_MEDIUM = 0.50

def compute_confidence_score(result: dict, ambiguity_info: dict) -> dict:
    """
    Composite confidence score built from four components:

        vector_similarity   — how close the retrieved policy is semantically
        constraint_match    — was a deterministic rule used? (hard signal)
        number_parse        — was normalization needed? (soft signal)
        ambiguity_penalty   — penalty for multiple / suspicious numbers
    """
    best = result.get("best_match")
    if not best:
        return {"score": 0.0, "components": {}, "level": "low"}

    # Chroma returns L2 distance; convert to similarity in [0, 1]
    raw_distance      = best.get("distance", 1.0)
    vector_similarity = max(0.0, 1.0 - min(raw_distance / 2.0, 1.0))

    constraint_score  = 1.0 if result.get("constraint_used") else 0.4

    number_parse_score = (
        0.7 if ambiguity_info["was_normalized"]
        else (0.5 if ambiguity_info["suspicious_small"] else 1.0)
    )

    ambiguity_penalty = 0.1 * int(ambiguity_info["multiple_numbers"])

    score = (
        0.35 * vector_similarity
      + 0.40 * constraint_score
      + 0.20 * number_parse_score
      - ambiguity_penalty
    )
    score = round(max(0.0, min(1.0, score)), 3)

    if score >= CONFIDENCE_THRESHOLD_HIGH:
        level = "high"
    elif score >= CONFIDENCE_THRESHOLD_MEDIUM:
        level = "medium"
    else:
        level = "low"

    return {
        "score": score,
        "level": level,
        "components": {
            "vector_similarity" : round(vector_similarity, 3),
            "constraint_match"  : constraint_score,
            "number_parse"      : number_parse_score,
            "ambiguity_penalty" : ambiguity_penalty,
        }
    }


# ─────────────────────────────────────────────
# 4️⃣  Decision Policy
# ─────────────────────────────────────────────

def make_confidence_decision(confidence: dict) -> str:
    """
    Map confidence level to a safe action.

        high    → auto_decision
        medium  → ask_clarification
        low     → escalate_to_human
    """
    mapping = {
        "high"  : "auto_decision",
        "medium": "ask_clarification",
        "low"   : "escalate_to_human",
    }
    return mapping.get(confidence["level"], "escalate_to_human")


# ─────────────────────────────────────────────
# 🔗  Full Production Pipeline
# ─────────────────────────────────────────────

def production_decision_pipeline(raw_query: str, k: int = 5) -> dict:
    """
    End-to-end production pipeline:

        raw_query
            ↓ normalize_monetary_input
            ↓ hybrid_policy_retrieval
            ↓ detect_ambiguity
            ↓ compute_confidence_score
            ↓ make_confidence_decision
            ↓ build_decision_output
    """

    # Step 1 — Normalize
    normalized_query, was_normalized = normalize_monetary_input(raw_query)

    # Step 2 — Retrieve (always run on the normalized query)
    result = hybrid_policy_retrieval(normalized_query, k=k)

    # Step 3 — Detect ambiguity
    ambiguity = detect_ambiguity(raw_query, normalized_query, was_normalized)

    # Step 4 — Score confidence
    confidence = compute_confidence_score(result, ambiguity)

    # Step 5 — Action
    action = make_confidence_decision(confidence)

    # Step 6 — Build structured decision (only if safe to act)
    decision_output = (
        build_decision_output(result, normalized_query)
        if action == "auto_decision"
        else {"decision": None, "reason": f"Action required: {action}"}
    )

    return {
        "raw_query"       : raw_query,
        "normalized_query": normalized_query,
        "ambiguity"       : ambiguity,
        "confidence"      : confidence,
        "action"          : action,
        "decision_output" : decision_output,
        "timestamp"       : datetime.now(timezone.utc).isoformat(),
    }

Demo

In [65]:
test_queries = [
    "Who signs off on a 4k hardware purchase?",        # ambiguous — needs normalization
    "Purchase device with price $4,000 who approves?", # formatted number
    "I need to buy a 4 thousand dollar laptop",        # written-out form
    "Purchase device with price 4000$ who approves?",  # clean — baseline
    "Can I buy something for 8 dollars?",              # suspiciously small
]

for q in test_queries:
    output = production_decision_pipeline(q)

    print("=" * 65)
    print(f"  Query      : {output['raw_query']}")
    print(f"  Normalized : {output['normalized_query']}")
    print(f"  Ambiguous  : {output['ambiguity']['is_ambiguous']}  "
          f"| k-pattern: {output['ambiguity']['k_pattern_found']}  "
          f"| normalized numbers: {output['ambiguity']['normalized_numbers']}")
    print(f"  Confidence : {output['confidence']['score']}  "
          f"({output['confidence']['level'].upper()})")
    print(f"  Components : {output['confidence']['components']}")
    print(f"  Action     : {output['action'].upper()}")

    if output["action"] == "auto_decision":
        d = output["decision_output"].get("decision", {})
        print(f"  Decision   : approval={d.get('approval_required')}  "
              f"risk={d.get('risk_level')}")
    else:
        print(f"  Decision   : {output['decision_output']['reason']}")

print("=" * 65)

  Query      : Who signs off on a 4k hardware purchase?
  Normalized : Who signs off on a 4000 hardware purchase?
  Ambiguous  : True  | k-pattern: True  | normalized numbers: [4000.0]
  Confidence : 0.71  (MEDIUM)
  Components : {'vector_similarity': 0.486, 'constraint_match': 1.0, 'number_parse': 0.7, 'ambiguity_penalty': 0.0}
  Action     : ASK_CLARIFICATION
  Decision   : Action required: ask_clarification
  Query      : Purchase device with price $4,000 who approves?
  Normalized : Purchase device with price 4000 who approves?
  Ambiguous  : True  | k-pattern: False  | normalized numbers: [4000.0]
  Confidence : 0.732  (MEDIUM)
  Components : {'vector_similarity': 0.549, 'constraint_match': 1.0, 'number_parse': 0.7, 'ambiguity_penalty': 0.0}
  Action     : ASK_CLARIFICATION
  Decision   : Action required: ask_clarification
  Query      : I need to buy a 4 thousand dollar laptop
  Normalized : I need to buy a 4000 dollar laptop
  Ambiguous  : True  | k-pattern: False  | normalized 

#Lab 3— Tasks & Extensions

## Task 1 — Intelligent Section Routing using LLM



### Objective

Improve the Hybrid Compliance System.

Instead of manually writing:

```python
where={"section": "Financial"}
```

The system must automatically detect the correct section using the LLM.

The LLM will act as a **section classifier**.

---

# What You Must Build

You must create a new function:

```python
def detect_section_llm(question: str) -> dict:
```

This function must:

* Take the user question
* Send it to the LLM
* Return a JSON object
* Contain only the detected section

---

# Allowed Sections

The LLM is allowed to return ONLY one of these values:

* Financial
* HR
* Legal
* Security
* Unknown

If unsure → return `"Unknown"`

---

# Required Output Format (STRICT)

The LLM must return JSON only:

```json
{"section": "Financial"}
```

⚠ No extra text.
⚠ No explanations.
⚠ No markdown.
⚠ JSON only.

---

# Integration Requirement

After building the classifier, integrate it into the system:

```python
section = detect_section_llm(question)

result = hybrid_policy_retrieval(
    question,
    k=5,
    where=section if section["section"] != "Unknown" else None
)
```

If the section is `"Unknown"`, do NOT apply metadata filtering.

---

# Safety Requirements

You must:

* Validate the JSON output
* Ensure the section is one of the allowed labels
* Handle JSON parsing errors
* Fail safely (fallback to no filter)

---

# Testing Requirements

Test at least:

* 3 Financial queries
* 3 HR queries
* 3 Legal queries
* 2 ambiguous queries

Example:

* "Purchase device with price 4000$"
* "How many sick days do employees get?"
* "What are the penalties for contract breach?"

---

# Bonus (Optional but Recommended)

Compare performance:

* Without LLM section routing
* With LLM section routing

Measure:

* Latency
* Recall@K
* Constraint Accuracy

Explain the difference.

---

# Learning Outcome

After this task, you should understand:

* LLM as a classifier
* Structured output enforcement
* LLM + deterministic filtering
* Safe integration of LLM into production systems
* Controlled hybrid AI architecture

---

# Advanced Bonus (Optional)

Add a confidence score for section detection:

```json
{"section": "Financial", "confidence": 0.82}
```

Use confidence to decide whether to apply metadata filtering.

---

🔥 This is not a prompt exercise.
This is a system design improvement task.


detect_section_llm (Section Classifier)

In [66]:
import json
import re
import time
from transformers import pipeline

# ─────────────────────────────────────────────────────────────
# Load Phi-3.5 from local Google Drive path
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "/content/drive/MyDrive/Phi_3_5_mini_instruct"

try:
    llm_pipe
    print("✅ Reusing existing LLM pipeline")
except NameError:
    print(f"⏳ Loading Phi-3.5-mini-instruct from {MODEL_PATH} ...")
    llm_pipe = pipeline(
        "text-generation",
        model=MODEL_PATH,           # ← local path instead of HuggingFace ID
        trust_remote_code=True,
        device_map="auto",
    )
    print("✅ LLM pipeline ready")

# ─────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────
ALLOWED_SECTIONS = {"Financial", "HR", "Legal", "Security", "Unknown"}

SECTION_SYSTEM_PROMPT = """You are a compliance section classifier.

Your ONLY job is to read the user question and return the correct section label.

Allowed labels:
- Financial   → purchases, budgets, approvals, invoices, expenses
- HR          → employees, leave, sick days, hiring, performance, salary
- Legal       → contracts, penalties, compliance, lawsuits, agreements
- Security    → data, access, passwords, encryption, software, systems
- Unknown     → if you are not sure

Rules:
- Return ONLY a JSON object.
- Do NOT add any explanation, text, or markdown.
- Do NOT use code blocks.
- Always include a confidence score between 0.0 and 1.0.

Required output format:
{"section": "Financial", "confidence": 0.92}"""


# ─────────────────────────────────────────────────────────────
# Core classifier
# ─────────────────────────────────────────────────────────────
def detect_section_llm(question: str) -> dict:
    """
    Use the LLM as a section classifier.

    Returns:
        {
            "section"    : "Financial",   # one of ALLOWED_SECTIONS
            "confidence" : 0.92,          # float in [0.0, 1.0]
            "raw_output" : "...",         # raw LLM response (for debugging)
            "fallback"   : False          # True if JSON parsing failed
        }
    """

    messages = [
        {"role": "system", "content": SECTION_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Question: {question}"}
    ]

    # ── Call LLM ──────────────────────────────────────────────
    try:
        output = llm_pipe(
            messages,
            max_new_tokens=60,       # section JSON is tiny
            do_sample=False,         # deterministic output
            temperature=None,
            top_p=None,
            return_full_text=False,
        )
        raw = output[0]["generated_text"].strip()
    except Exception as e:
        print(f"⚠️  LLM call failed: {e}")
        return _fallback_response(reason=f"LLM error: {e}")

    # ── Parse JSON ────────────────────────────────────────────
    return _parse_and_validate(raw)


def _parse_and_validate(raw: str) -> dict:
    """
    Parse the LLM output and validate the section label.
    Falls back to Unknown if parsing fails.
    """
    # Extract first JSON-like object in case of extra text
    match = re.search(r"\{.*?\}", raw, re.DOTALL)
    json_str = match.group(0) if match else raw

    try:
        parsed = json.loads(json_str)
    except json.JSONDecodeError:
        print(f"⚠️  JSON parse error. Raw output: {repr(raw)}")
        return _fallback_response(reason="JSON parse error", raw=raw)

    # ── Validate section label ────────────────────────────────
    section    = parsed.get("section", "Unknown")
    confidence = parsed.get("confidence", 0.5)

    if section not in ALLOWED_SECTIONS:
        print(f"⚠️  Invalid section '{section}' → falling back to Unknown")
        section    = "Unknown"
        confidence = 0.0

    # ── Clamp confidence ──────────────────────────────────────
    try:
        confidence = float(confidence)
        confidence = max(0.0, min(1.0, confidence))
    except (ValueError, TypeError):
        confidence = 0.5

    return {
        "section"    : section,
        "confidence" : round(confidence, 3),
        "raw_output" : raw,
        "fallback"   : False
    }


def _fallback_response(reason: str = "unknown", raw: str = "") -> dict:
    return {
        "section"    : "Unknown",
        "confidence" : 0.0,
        "raw_output" : raw,
        "fallback"   : True,
        "reason"     : reason
    }


# ─────────────────────────────────────────────────────────────
# Confidence-aware section filter
# ─────────────────────────────────────────────────────────────
SECTION_CONFIDENCE_THRESHOLD = 0.6   # below this → no filter applied

def section_to_filter(section_result: dict) -> dict | None:
    """
    Convert section detection result to a Chroma 'where' filter.

    Returns None if:
        - section is Unknown
        - confidence is below threshold
    """
    if section_result["section"] == "Unknown":
        return None
    if section_result["confidence"] < SECTION_CONFIDENCE_THRESHOLD:
        print(f"⚠️  Low confidence ({section_result['confidence']}) → skipping metadata filter")
        return None
    return {"section": section_result["section"]}

⏳ Loading Phi-3.5-mini-instruct from /content/drive/MyDrive/Phi_3_5_mini_instruct ...


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ LLM pipeline ready


Integration + Smart Retrieval

In [67]:
def smart_hybrid_retrieval(question: str, k: int = 5) -> dict:
    """
    Full pipeline with LLM section routing:

        question
            ↓ detect_section_llm        (LLM classifier)
            ↓ section_to_filter         (confidence gate)
            ↓ hybrid_policy_retrieval   (Vector DB + constraint engine)
    """
    t0 = time.time()

    # Step 1 — LLM section detection
    section_result = detect_section_llm(question)

    # Step 2 — Confidence gate → where filter (or None)
    where_filter = section_to_filter(section_result)

    # Step 3 — Hybrid retrieval with auto-detected filter
    result = hybrid_policy_retrieval(question, k=k, where=where_filter)

    latency_ms = round((time.time() - t0) * 1000, 1)

    return {
        "question"           : question,
        "detected_section"   : section_result["section"],
        "section_confidence" : section_result["confidence"],
        "filter_applied"     : where_filter,
        "constraint_used"    : result["constraint_used"],
        "best_match"         : result["best_match"],
        "candidates"         : result["candidates"],
        "latency_ms"         : latency_ms,
    }


# ─────────────────────────────────────────────────────────────
# Test suite: 3 Financial + 3 HR + 3 Legal + 2 Ambiguous
# ─────────────────────────────────────────────────────────────
test_queries = [
    # Financial
    {"query": "Purchase device with price 4000$, who approves?",       "expected": "Financial"},
    {"query": "Do I need approval for a $500 software subscription?",  "expected": "Financial"},
    {"query": "Who signs off on a hardware budget above $3000?",       "expected": "Financial"},
    # HR
    {"query": "How many sick days do employees get per year?",         "expected": "HR"},
    {"query": "What is the remote work policy for new hires?",         "expected": "HR"},
    {"query": "What is the process for requesting annual leave?",      "expected": "HR"},
    # Legal
    {"query": "What are the penalties for contract breach?",           "expected": "Legal"},
    {"query": "Does the NDA cover third-party contractors?",           "expected": "Legal"},
    {"query": "What compliance rules apply to vendor agreements?",     "expected": "Legal"},
    # Ambiguous
    {"query": "Can I share the report with the client?",               "expected": "Unknown"},
    {"query": "What do I need to do before the meeting?",              "expected": "Unknown"},
]

print("=" * 70)
print(f"{'Query':<48} {'Expected':<12} {'Detected':<12} {'Conf':<6} {'✓'}")
print("=" * 70)

correct = 0
for item in test_queries:
    result = smart_hybrid_retrieval(item["query"], k=5)

    detected  = result["detected_section"]
    conf      = result["section_confidence"]
    expected  = item["expected"]
    match     = "✅" if detected == expected else "❌"

    if detected == expected:
        correct += 1

    print(f"{item['query'][:47]:<48} {expected:<12} {detected:<12} {conf:<6} {match}")
    print(f"  → Filter: {result['filter_applied']}  |  "
          f"Constraint used: {result['constraint_used']}  |  "
          f"Latency: {result['latency_ms']} ms")
    print()

accuracy = correct / len(test_queries)
print("=" * 70)
print(f"Section Routing Accuracy: {correct}/{len(test_queries)} = {accuracy:.1%}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/

Query                                            Expected     Detected     Conf   ✓
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
Purchase device with price 4000$, who approves?  Financial    Unknown      0.0    ❌
  → Filter: None  |  Constraint used: True  |  Latency: 111.5 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
Do I need approval for a $500 software subscrip  Financial    Unknown      0.0    ❌
  → Filter: None  |  Constraint used: True  |  Latency: 37.7 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Who signs off on a hardware budget above $3000?  Financial    Unknown      0.0    ❌
  → Filter: None  |  Constraint used: True  |  Latency: 35.0 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

How many sick days do employees get per year?    HR           Unknown      0.0    ❌
  → Filter: None  |  Constraint used: False  |  Latency: 38.7 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
What is the remote work policy for new hires?    HR           Unknown      0.0    ❌
  → Filter: None  |  Constraint used: False  |  Latency: 43.3 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
What is the process for requesting annual leave  HR           Unknown      0.0    ❌
  → Filter: None  |  Constraint used: False  |  Latency: 38.2 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
What are the penalties for contract breach?      Legal        Unknown      0.0    ❌
  → Filter: None  |  Constraint used: False  |  Latency: 32.7 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
Does the NDA cover third-party contractors?      Legal      

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What compliance rules apply to vendor agreement  Legal        Unknown      0.0    ❌
  → Filter: None  |  Constraint used: False  |  Latency: 40.0 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Can I share the report with the client?          Unknown      Unknown      0.0    ✅
  → Filter: None  |  Constraint used: False  |  Latency: 34.3 ms

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
What do I need to do before the meeting?         Unknown      Unknown      0.0    ✅
  → Filter: None  |  Constraint used: False  |  Latency: 27.8 ms

Section Routing Accuracy: 2/11 = 18.2%


Bonus: Performance Comparison (With vs Without Routing)

In [68]:
import time

# Use the 20 Financial queries already defined in the notebook
# (evaluation_queries_20 — all are Financial section)

def evaluate_with_routing(eval_data, k=5):
    total = len(eval_data)
    constraint_correct = 0
    recall_hits = 0
    total_latency = 0

    for item in eval_data:
        query    = item["query"]
        expected = item["expected_condition"]

        result = smart_hybrid_retrieval(query, k=k)
        total_latency += result["latency_ms"]

        best = result["best_match"]
        if best:
            if best["metadata"]["condition"] == expected:
                constraint_correct += 1

        # Recall@K
        for h in result["candidates"]:
            if h["metadata"]["condition"] == expected:
                recall_hits += 1
                break

    return {
        "System"              : "Hybrid + LLM Routing",
        "Total Queries"       : total,
        "Recall@K"            : round(recall_hits / total, 3),
        "Constraint Accuracy" : round(constraint_correct / total, 3),
        "Avg Latency (ms)"    : round(total_latency / total, 1),
    }


# ── Run both systems ──────────────────────────────────────────
print("⏳ Evaluating Hybrid (no routing) ...")
baseline = evaluate_system(evaluation_queries_20, k=5)
baseline["System"] = "Hybrid (no routing)"

print("⏳ Evaluating Hybrid + LLM Section Routing ...")
with_routing = evaluate_with_routing(evaluation_queries_20, k=5)

# ── Print comparison table ────────────────────────────────────
metrics = ["System", "Total Queries", "Recall@K", "Constraint Accuracy", "Avg Latency (ms)"]

print("\n" + "=" * 70)
for m in metrics:
    b_val = baseline.get(m, baseline.get("Average Latency (ms)", "-"))
    r_val = with_routing.get(m, "-")

    # Rename key mismatch
    if m == "Avg Latency (ms)":
        b_val = round(baseline.get("Average Latency (ms)", 0), 1)

    print(f"  {m:<25} {str(b_val):<25} {str(r_val)}")

print("=" * 70)

print("""
Analysis:
─────────────────────────────────────────────────────────────────────
Constraint Accuracy  → Should be identical (both use the same rule engine)
                       LLM routing narrows the search space, not the logic.

Recall@K             → With routing applied on Financial queries, Recall@K
                       stays high because all correct policies are Financial.
                       On cross-section queries, routing prevents false positives
                       from unrelated sections appearing in top-k.

Latency              → LLM routing adds overhead (Phi inference time).
                       Trade-off: slower per query, but more precise filtering
                       in multi-section databases.

Key Insight:
  No routing   → Vector search covers all sections (more noise in top-k)
  LLM routing  → Search scoped to detected section (less noise, cleaner top-k)
  The Constraint Engine makes the final decision in both cases.
""")

⏳ Evaluating Hybrid (no routing) ...


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

⏳ Evaluating Hybrid + LLM Section Routing ...
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'

  System                    Hybrid (no routing)       Hybrid + LLM Routing
  Total Queries             20                        20
  Recall@K                  1.0                       1.0
  Constraint Accuracy       1.0                       1.0
  Avg Latency (ms)          25.8                      29.2

Analysis:
─────────────────────────────────────────────────────────────────────
Constraint Accuracy  → Should be identical (both use the same rule engine)
                       LLM

**Architecture summary of what was built:**
```
User Question
      ↓
detect_section_llm()          ← LLM classifier (Phi-3.5)
      ↓                           Returns: {"section": "Financial", "confidence": 0.92}
section_to_filter()           ← Confidence gate (threshold = 0.60)
      ↓                           Returns: {"section": "Financial"} OR None
hybrid_policy_retrieval()     ← Vector DB + constraint engine
      ↓
build_decision_output()       ← Structured decision

##  Task 2 — Add Confidence & Escalation Layer (Enterprise Thinking)



### 🎯 Objective

Upgrade the Hybrid Compliance System to behave like a real enterprise system.

The system must NOT always auto-decide.
It must decide only when confidence is high.

---

# 🏗 What You Must Build

You must add a **confidence scoring layer** after policy selection.

The system should calculate a numeric confidence score between 0 and 1.

---

# 🧠 Confidence Factors

You must combine at least 3 of the following signals:

* Vector similarity score
* Whether constraint matching was used (True/False)
* Section routing confidence (if implemented)
* Number parsing reliability
* Metadata match strength

Example logic:

```python
confidence = (
    similarity_score * 0.4 +
    (1 if constraint_used else 0) * 0.3 +
    routing_confidence * 0.3
)
```

⚠ You must explain your weighting strategy.

---

# 🧭 Decision Policy

Based on the confidence score, implement this policy:

| Confidence | Action            |
| ---------- | ----------------- |
| > 0.75     | AUTO_DECISION     |
| 0.5–0.75   | ASK_CLARIFICATION |
| < 0.5      | ESCALATE_TO_HUMAN |

---

# 📤 Required Output Format

Your system must now return structured output like:

```json
{
  "decision": "REQUIRES_APPROVAL",
  "confidence_score": 0.82,
  "action": "AUTO_DECISION",
  "reason": "High similarity and valid constraint match"
}
```

If escalated:

```json
{
  "decision": null,
  "confidence_score": 0.41,
  "action": "ESCALATE_TO_HUMAN",
  "reason": "Low similarity and ambiguous numeric input"
}
```

---

# 🛡 Safety Requirements

You must:

* Prevent auto-decision if confidence is low
* Handle missing values safely
* Document your threshold choices

---

# 🧪 Testing Requirements

Test at least:

* 3 high-confidence cases
* 3 medium-confidence cases
* 3 low-confidence cases
* 2 ambiguous input cases (example: "4k hardware purchase")

---

# 🧠 Learning Outcome

After this task, you should understand:

* AI risk control
* Confidence-based decision making
* Enterprise-grade AI behavior
* Safe automation boundaries

---


Confidence Scoring Layer

In [69]:
# ═══════════════════════════════════════════════════════════════
# TASK 2 — Confidence & Escalation Layer
#
# Weighting Strategy (must sum to 1.0):
#
#   vector_similarity   → 0.35  How semantically close is the retrieved policy?
#                                Core retrieval quality signal.
#
#   constraint_match    → 0.35  Was a deterministic rule used?
#                                Strongest signal — rule match = logically correct.
#
#   section_routing     → 0.20  How confident was the LLM section classifier?
#                                Scoped retrieval = less noise in top-k.
#
#   metadata_strength   → 0.10  Does the policy section match the detected section?
#                                Lightweight cross-validation check.
#
# Thresholds:
#   > 0.75  → AUTO_DECISION      (safe to act automatically)
#   0.5–0.75 → ASK_CLARIFICATION (uncertain — ask user to confirm)
#   < 0.5   → ESCALATE_TO_HUMAN  (too risky — hand off to a person)
# ═══════════════════════════════════════════════════════════════

THRESHOLD_HIGH   = 0.75
THRESHOLD_MEDIUM = 0.50


def compute_confidence(
    best_match      : dict,
    constraint_used : bool,
    section_result  : dict,
) -> dict:
    """
    Compute a composite confidence score from 4 signals.

    Parameters
    ----------
    best_match      : the top policy dict returned by hybrid_policy_retrieval
    constraint_used : whether the deterministic rule engine fired
    section_result  : output of detect_section_llm  (section + confidence)

    Returns
    -------
    {
        "score"      : float   # final weighted score in [0.0, 1.0]
        "level"      : str     # "high" | "medium" | "low"
        "components" : dict    # individual signal scores for explainability
    }
    """

    # ── Signal 1: Vector Similarity ───────────────────────────
    # Chroma returns L2 distance. Convert to similarity in [0, 1].
    # distance=0 → perfect match → similarity=1.0
    # distance=2 → worst case    → similarity=0.0
    raw_distance      = best_match.get("distance", 2.0)
    vector_similarity = round(max(0.0, 1.0 - min(raw_distance / 2.0, 1.0)), 3)

    # ── Signal 2: Constraint Match ─────────────────────────────
    # Binary: did the rule engine fire?
    # 1.0 = deterministic rule matched (logically correct)
    # 0.3 = fell back to vector similarity only (weaker guarantee)
    constraint_score = 1.0 if constraint_used else 0.3

    # ── Signal 3: Section Routing Confidence ──────────────────
    # LLM classifier confidence score already in [0.0, 1.0]
    # If Task 1 was not run, defaults to 0.5 (neutral)
    routing_confidence = float(section_result.get("confidence", 0.5))

    # ── Signal 4: Metadata Match Strength ─────────────────────
    # Does the retrieved policy's section match the detected section?
    # Cheap cross-validation: if both agree → higher trust
    detected_section  = section_result.get("section", "Unknown")
    policy_section    = best_match.get("metadata", {}).get("section", "")

    if detected_section == "Unknown":
        metadata_score = 0.5   # neutral — no routing was applied
    elif detected_section == policy_section:
        metadata_score = 1.0   # perfect agreement
    else:
        metadata_score = 0.1   # mismatch — likely wrong policy retrieved

    # ── Weighted combination ───────────────────────────────────
    score = (
        vector_similarity  * 0.35 +
        constraint_score   * 0.35 +
        routing_confidence * 0.20 +
        metadata_score     * 0.10
    )
    score = round(max(0.0, min(1.0, score)), 3)

    # ── Level classification ───────────────────────────────────
    if score > THRESHOLD_HIGH:
        level = "high"
    elif score >= THRESHOLD_MEDIUM:
        level = "medium"
    else:
        level = "low"

    return {
        "score"  : score,
        "level"  : level,
        "components": {
            "vector_similarity"  : vector_similarity,
            "constraint_match"   : constraint_score,
            "section_routing"    : round(routing_confidence, 3),
            "metadata_strength"  : metadata_score,
        }
    }


# ─────────────────────────────────────────────────────────────
# Decision Policy
# ─────────────────────────────────────────────────────────────

def make_action(level: str) -> str:
    return {
        "high"  : "AUTO_DECISION",
        "medium": "ASK_CLARIFICATION",
        "low"   : "ESCALATE_TO_HUMAN",
    }[level]


def build_reason(confidence: dict, constraint_used: bool, section_result: dict) -> str:
    parts = []

    c = confidence["components"]

    if c["vector_similarity"] >= 0.6:
        parts.append("high semantic similarity")
    elif c["vector_similarity"] >= 0.3:
        parts.append("moderate semantic similarity")
    else:
        parts.append("low semantic similarity")

    if constraint_used:
        parts.append("deterministic rule matched")
    else:
        parts.append("no rule matched — vector fallback used")

    if section_result.get("section") != "Unknown":
        parts.append(f"section routed to '{section_result['section']}' "
                     f"(conf={c['section_routing']})")
    else:
        parts.append("section unknown — no metadata filter applied")

    if c["metadata_strength"] == 1.0:
        parts.append("policy section confirmed")
    elif c["metadata_strength"] == 0.1:
        parts.append("policy section mismatch detected")

    return ", ".join(parts)

 Full Enterprise Decision Pipeline

In [70]:
def enterprise_decision(question: str, k: int = 5) -> dict:
    """
    End-to-end enterprise compliance pipeline:

        question
            ↓  detect_section_llm          Task 1 — LLM section classifier
            ↓  hybrid_policy_retrieval      Hybrid retrieval + rule engine
            ↓  compute_confidence           Task 2 — 4-signal confidence score
            ↓  make_action                  Task 2 — escalation policy
            ↓  structured output            Auditable, explainable result
    """
    from datetime import datetime, timezone

    # ── Step 1: LLM section routing ───────────────────────────
    section_result = detect_section_llm(question)
    where_filter   = section_to_filter(section_result)

    # ── Step 2: Hybrid retrieval ──────────────────────────────
    retrieval = hybrid_policy_retrieval(question, k=k, where=where_filter)
    best      = retrieval.get("best_match")

    # ── Step 3: No policy found → hard escalation ─────────────
    if not best:
        return {
            "question"        : question,
            "decision"        : None,
            "confidence_score": 0.0,
            "action"          : "ESCALATE_TO_HUMAN",
            "reason"          : "No matching policy found in the database",
            "policy"          : None,
            "section_routing" : section_result,
            "timestamp"       : datetime.now(timezone.utc).isoformat(),
        }

    # ── Step 4: Confidence scoring ────────────────────────────
    confidence = compute_confidence(
        best_match      = best,
        constraint_used = retrieval["constraint_used"],
        section_result  = section_result,
    )

    # ── Step 5: Action decision ───────────────────────────────
    action = make_action(confidence["level"])
    reason = build_reason(confidence, retrieval["constraint_used"], section_result)

    # ── Step 6: Decision value ─────────────────────────────────
    # Only expose the actual decision if confidence is high enough
    if action == "AUTO_DECISION":
        approval = best["metadata"].get("approval_required")
        decision_value = "AUTO_APPROVED" if approval == "No" else "REQUIRES_APPROVAL"
    else:
        decision_value = None   # do NOT auto-decide under uncertainty

    return {
        "question"         : question,
        "decision"         : decision_value,
        "approval_required": best["metadata"].get("approval_required") if action == "AUTO_DECISION" else None,
        "risk_level"       : best["metadata"].get("risk_level"),
        "confidence_score" : confidence["score"],
        "confidence_level" : confidence["level"],
        "confidence_detail": confidence["components"],
        "action"           : action,
        "reason"           : reason,
        "constraint_used"  : retrieval["constraint_used"],
        "policy_matched"   : best["metadata"].get("policy_name"),
        "section_routing"  : {
            "detected" : section_result["section"],
            "confidence": section_result["confidence"],
            "filter_applied": where_filter,
        },
        "timestamp"        : datetime.now(timezone.utc).isoformat(),
    }


def print_enterprise_result(result: dict):
    """Pretty-print one enterprise decision result."""
    action_icon = {"AUTO_DECISION": "✅", "ASK_CLARIFICATION": "⚠️", "ESCALATE_TO_HUMAN": "🚨"}
    icon = action_icon.get(result["action"], "❓")

    print(f"  Question   : {result['question']}")
    print(f"  Action     : {icon}  {result['action']}")
    print(f"  Decision   : {result['decision']}")
    print(f"  Approval   : {result['approval_required']}")
    print(f"  Confidence : {result['confidence_score']}  ({result['confidence_level'].upper()})")
    print(f"  Components : {result['confidence_detail']}")
    print(f"  Reason     : {result['reason']}")
    print(f"  Policy     : {result['policy_matched']}")
    print(f"  Section    : {result['section_routing']['detected']}  "
          f"(conf={result['section_routing']['confidence']})")

Test Suite

In [71]:
# ═══════════════════════════════════════════════════════════════
# Test cases — grouped by expected confidence tier
# ═══════════════════════════════════════════════════════════════

test_cases = {

    # ── High confidence ────────────────────────────────────────
    # Clear section, numeric value, deterministic rule should fire
    "HIGH": [
        "Purchase device with price 4000$, who approves?",
        "Do I need approval for a $5000 server purchase?",
        "Who signs off on a hardware purchase of $1200?",
    ],

    # ── Medium confidence ──────────────────────────────────────
    # Clear intent but no numeric value → constraint engine won't fire
    "MEDIUM": [
        "Who approves hardware purchases above budget?",
        "What is the approval process for buying equipment?",
        "Can a manager approve a software subscription?",
    ],

    # ── Low confidence ─────────────────────────────────────────
    # Vague queries with no clear section or numeric signal
    "LOW": [
        "What do I need to do before the meeting?",
        "Can I share this file with someone?",
        "Is this allowed?",
    ],

    # ── Ambiguous monetary input ───────────────────────────────
    # These test the normalization + confidence interaction
    "AMBIGUOUS": [
        "Who signs off on a 4k hardware purchase?",
        "Approval needed for 4k and 300 equipment order?",
    ],
}

# ─────────────────────────────────────────────────────────────
# Run all tests
# ─────────────────────────────────────────────────────────────
summary = []

for tier, queries in test_cases.items():
    print(f"\n{'═'*65}")
    print(f"  {tier} CONFIDENCE CASES")
    print(f"{'═'*65}")

    for q in queries:
        result = enterprise_decision(q)
        print_enterprise_result(result)
        print(f"  {'─'*60}")

        summary.append({
            "tier"      : tier,
            "question"  : q[:50],
            "score"     : result["confidence_score"],
            "level"     : result["confidence_level"],
            "action"    : result["action"],
        })

# ─────────────────────────────────────────────────────────────
# Summary table
# ─────────────────────────────────────────────────────────────
print(f"\n{'═'*65}")
print(f"  SUMMARY")
print(f"{'═'*65}")
print(f"  {'Tier':<10} {'Score':<8} {'Level':<10} {'Action':<22} {'Query'}")
print(f"  {'─'*63}")
for r in summary:
    print(f"  {r['tier']:<10} {r['score']:<8} {r['level']:<10} {r['action']:<22} {r['question']}")

print(f"\n{'═'*65}")
print("  THRESHOLD DOCUMENTATION")
print(f"{'═'*65}")
print("""
  > 0.75  → AUTO_DECISION
            All 4 signals are strong. Rule engine fired on a specific number,
            section was detected with high confidence, and the retrieved policy
            section matched. Safe to act without human review.

  0.5–0.75 → ASK_CLARIFICATION
            Partial confidence. Usually means semantic match is good but no
            numeric constraint fired (no number in the query), or section
            routing was uncertain. Present decision to user for confirmation.

  < 0.5   → ESCALATE_TO_HUMAN
            Query is vague, section unknown, or semantic distance is large.
            The system has no reliable signal to base a decision on.
            A human must review this case.

  Ambiguous inputs ("4k"):
            Even after normalization, a conflict between multiple numbers
            or a suspicious small parsed value reduces constraint_match
            and vector_similarity scores, pushing confidence below 0.75
            and triggering ASK_CLARIFICATION or ESCALATE_TO_HUMAN.
""")

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma


═════════════════════════════════════════════════════════════════
  HIGH CONFIDENCE CASES
═════════════════════════════════════════════════════════════════
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
  Question   : Purchase device with price 4000$, who approves?
  Action     : ⚠️  ASK_CLARIFICATION
  Decision   : None
  Approval   : None
  Confidence : 0.575  (MEDIUM)
  Components : {'vector_similarity': 0.501, 'constraint_match': 1.0, 'section_routing': 0.0, 'metadata_strength': 0.5}
  Reason     : moderate semantic similarity, deterministic rule matched, section unknown — no metadata filter applied
  Policy     : Hardware Purchase Policy
  Section    : Unknown  (conf=0.0)
  ────────────────────────────────────────────────────────────
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
  Question   : Do I need approval for a $5000 server purchase?
  Action     : ⚠️  ASK_CLARIFICATION
  Decision   : None
  Appro

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Question   : What do I need to do before the meeting?
  Action     : 🚨  ESCALATE_TO_HUMAN
  Decision   : None
  Approval   : None
  Confidence : 0.221  (LOW)
  Components : {'vector_similarity': 0.189, 'constraint_match': 0.3, 'section_routing': 0.0, 'metadata_strength': 0.5}
  Reason     : low semantic similarity, no rule matched — vector fallback used, section unknown — no metadata filter applied
  Policy     : Travel Expense Policy
  Section    : Unknown  (conf=0.0)
  ────────────────────────────────────────────────────────────
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
  Question   : Can I share this file with someone?
  Action     : 🚨  ESCALATE_TO_HUMAN
  Decision   : None
  Approval   : None
  Confidence : 0.264  (LOW)
  Components : {'vector_similarity': 0.312, 'constraint_match': 0.3, 'section_routing': 0.0, 'metadata_strength': 0.5}
  Reason     : moderate semantic similarity, no rule matched — vector fallback used, section unknown —

##  Task 3 — Production Handling for Ambiguous Monetary Inputs (e.g., "4k")



### Objective

Upgrade the system to safely handle ambiguous monetary expressions.

Example user query:

> "Who signs off on a 4k hardware purchase?"

If the system parses "4k" as `4` instead of `4000`,
it may produce a logically incorrect decision.

This is NOT a retrieval problem.
This is a **normalization + confidence problem**.

---

# What You Must Build

You must implement a **Robust Monetary Normalization Layer** before numeric extraction.

---

## 1️⃣ Robust Normalization Layer

Normalize common monetary formats before extracting numbers.

Your system must correctly convert:

- 4k → 4000
- $4,000 → 4000
- 4 thousand → 4000
- USD 4k → 4000

⚠ Normalization must happen BEFORE calling `extract_numbers()`.

---

## 2️⃣ Ambiguity Detection

If parsing is uncertain, the system must NOT auto-decide.

Example ambiguity signals:

- Extracted value is suspiciously small
- Original query contains "k" but parsed value is low
- Multiple conflicting numbers detected
- No number extracted but financial intent detected

When ambiguity is detected:

→ Reduce confidence score
→ Trigger clarification or escalation

---

## 3️⃣ Confidence Integration

Number parsing must contribute to overall confidence.

Add a **number_parsing_confidence** component.

Example logic:

```python
if ambiguous_number:
    number_confidence = 0.2
else:
    number_confidence = 1.0
```

Combine it with your existing confidence scoring formula.

If total confidence < threshold:
→ Do NOT auto-approve.

---

## 4️⃣ Decision Policy

| Confidence Level | Action |
|------------------|--------|
| High             | AUTO_DECISION |
| Medium           | ASK_CLARIFICATION |
| Low              | ESCALATE_TO_HUMAN |

---

## Required Output Format

Your final decision output must now include:

```json
{
  "decision": "REQUIRES_APPROVAL",
  "confidence_score": 0.78,
  "number_parsing_confidence": 1.0,
  "action": "AUTO_DECISION"
}
```

If ambiguity detected:

```json
{
  "decision": null,
  "confidence_score": 0.42,
  "number_parsing_confidence": 0.2,
  "action": "ESCALATE_TO_HUMAN",
  "reason": "Ambiguous monetary input"
}
```

---

## Testing Requirements

Test at least:

- "4k hardware purchase"
- "$4,000 device approval"
- "USD 4k laptop"
- "4 thousand equipment"
- Conflicting input: "4k and 300 purchase"

---

## Learning Outcome

After this task, you should understand:

- Data normalization in AI systems
- Ambiguity-aware decision making
- Risk-aware automation
- Why parsing errors can break business logic

---

## Core Principle

Vector DB retrieves what is similar.  
Constraint engine selects what is logically correct.  
Confidence engine decides whether it is safe to act.

If uncertain → Never auto-decide.



Normalization + Ambiguity Detection

In [72]:
# ═══════════════════════════════════════════════════════════════
# TASK 3 — Monetary Normalization + Ambiguity Detection
#
# Why this matters:
#   extract_numbers("4k hardware purchase") → [4.0]   ← WRONG
#   extract_numbers("4000 hardware purchase") → [4000.0] ← CORRECT
#
# The constraint engine receives the number AFTER extraction.
# If normalization doesn't happen first, the rule engine gets 4
# instead of 4000 and silently selects the wrong policy.
# ═══════════════════════════════════════════════════════════════

import re


# ─────────────────────────────────────────────────────────────
# 1️⃣  Normalization Layer
# ─────────────────────────────────────────────────────────────

def normalize_monetary_query(query: str) -> tuple[str, bool]:
    """
    Normalize ambiguous monetary expressions BEFORE extract_numbers().

    Handles:
        4k          → 4000
        4.5k        → 4500
        USD 4k      → 4000
        $4,000      → 4000
        4 thousand  → 4000
        4 million   → 4000000

    Returns:
        (normalized_query, was_normalized)
        was_normalized=True signals that a transformation occurred
        and should reduce number_parsing_confidence slightly.
    """
    original  = query
    result    = query

    # Step 1 — Remove currency symbols and commas
    #   "$4,000"  →  "4000"
    result = re.sub(r"\$", "", result)
    result = re.sub(r"USD\s*", "", result, flags=re.IGNORECASE)
    result = re.sub(r",(?=\d)", "", result)          # remove commas inside numbers

    # Step 2 — Replace Xk / X.Yk  →  X*1000
    #   "4k"    → "4000"
    #   "4.5k"  → "4500"
    result = re.sub(
        r"(\d+(?:\.\d+)?)\s*k\b",
        lambda m: str(int(float(m.group(1)) * 1_000)),
        result,
        flags=re.IGNORECASE
    )

    # Step 3 — Replace X thousand  →  X*1000
    #   "4 thousand" → "4000"
    result = re.sub(
        r"(\d+(?:\.\d+)?)\s*thousand\b",
        lambda m: str(int(float(m.group(1)) * 1_000)),
        result,
        flags=re.IGNORECASE
    )

    # Step 4 — Replace X million  →  X*1000000
    result = re.sub(
        r"(\d+(?:\.\d+)?)\s*million\b",
        lambda m: str(int(float(m.group(1)) * 1_000_000)),
        result,
        flags=re.IGNORECASE
    )

    was_normalized = result.strip() != original.strip()
    return result, was_normalized


# ─────────────────────────────────────────────────────────────
# 2️⃣  Ambiguity Detection
# ─────────────────────────────────────────────────────────────

# Financial intent keywords — used to detect "no number but financial query"
FINANCIAL_INTENT_KEYWORDS = [
    "purchase", "buy", "price", "cost", "budget", "spend",
    "approve", "approval", "expense", "invoice", "pay",
    "hardware", "software", "subscription", "equipment",
]

def detect_ambiguity(
    original_query   : str,
    normalized_query : str,
    was_normalized   : bool,
) -> dict:
    """
    Detect signals that numeric parsing may be unreliable.

    Ambiguity signals checked:
        k_pattern_present  — query had "4k" style before normalization
        suspicious_small   — number after normalization is still < 10
        multiple_numbers   — more than one number found (conflicting values)
        no_number_but_fin  — no number found but financial intent detected

    Returns a detailed dict consumed by compute_confidence_v3().
    """
    original_nums  = extract_numbers(original_query)
    normalized_nums = extract_numbers(normalized_query)

    k_pattern_present = bool(
        re.search(r"\d+(?:\.\d+)?\s*k\b", original_query, re.IGNORECASE)
    )
    suspicious_small   = bool(normalized_nums) and min(normalized_nums) < 10
    multiple_numbers   = len(normalized_nums) > 1
    no_number_but_fin  = (
        len(normalized_nums) == 0 and
        any(kw in original_query.lower() for kw in FINANCIAL_INTENT_KEYWORDS)
    )

    is_ambiguous = (
        suspicious_small or
        multiple_numbers or
        no_number_but_fin or
        (k_pattern_present and not was_normalized)   # "k" found but not normalized
    )

    return {
        "is_ambiguous"       : is_ambiguous,
        "was_normalized"     : was_normalized,
        "k_pattern_present"  : k_pattern_present,
        "suspicious_small"   : suspicious_small,
        "multiple_numbers"   : multiple_numbers,
        "no_number_but_fin"  : no_number_but_fin,
        "original_numbers"   : original_nums,
        "normalized_numbers" : normalized_nums,
    }


# ─────────────────────────────────────────────────────────────
# 3️⃣  Number Parsing Confidence
# ─────────────────────────────────────────────────────────────

def compute_number_parsing_confidence(ambiguity: dict) -> float:
    """
    Assign a number_parsing_confidence score based on ambiguity signals.

    Score logic:
        1.0  — clean number, no normalization needed, no ambiguity
        0.8  — normalization happened (e.g. "4k") but result is unambiguous
        0.4  — multiple conflicting numbers detected
        0.2  — suspiciously small value after normalization
        0.1  — no number found but financial intent present
        0.0  — multiple signals firing simultaneously (worst case)
    """
    if not ambiguity["is_ambiguous"] and not ambiguity["was_normalized"]:
        return 1.0                          # clean input — fully reliable

    if ambiguity["was_normalized"] and not ambiguity["is_ambiguous"]:
        return 0.8                          # normalized cleanly — slightly less certain

    score = 1.0

    if ambiguity["suspicious_small"]:
        score -= 0.8
    if ambiguity["multiple_numbers"]:
        score -= 0.6
    if ambiguity["no_number_but_fin"]:
        score -= 0.9
    if ambiguity["k_pattern_present"] and not ambiguity["was_normalized"]:
        score -= 0.5

    return round(max(0.0, score), 3)

 Upgraded compute_confidence (adds 5th signal)

In [73]:
# ═══════════════════════════════════════════════════════════════
# Upgrade compute_confidence from Task 2
# Now includes number_parsing_confidence as the 5th signal.
#
# Updated Weighting Strategy:
#
#   vector_similarity      → 0.30  (was 0.35 — slightly reduced to fit 5th signal)
#   constraint_match       → 0.30  (was 0.35)
#   section_routing        → 0.20  (unchanged)
#   metadata_strength      → 0.10  (unchanged)
#   number_parsing         → 0.10  (NEW — guards against parse errors)
#
# Why number_parsing weight is 0.10:
#   It acts as a safety penalty rather than a primary signal.
#   A parsing failure alone shouldn't block a decision, but
#   combined with low constraint_match it will push score below 0.75.
# ═══════════════════════════════════════════════════════════════

def compute_confidence_v3(
    best_match              : dict,
    constraint_used         : bool,
    section_result          : dict,
    number_parsing_confidence: float,
) -> dict:
    """
    5-signal confidence scorer.

    Extends Task 2's compute_confidence() with number_parsing_confidence.
    Drop-in replacement — call this instead of compute_confidence().
    """

    # Signal 1 — Vector Similarity
    raw_distance      = best_match.get("distance", 2.0)
    vector_similarity = round(max(0.0, 1.0 - min(raw_distance / 2.0, 1.0)), 3)

    # Signal 2 — Constraint Match
    constraint_score  = 1.0 if constraint_used else 0.3

    # Signal 3 — Section Routing Confidence
    routing_confidence = float(section_result.get("confidence", 0.5))

    # Signal 4 — Metadata Match Strength
    detected_section = section_result.get("section", "Unknown")
    policy_section   = best_match.get("metadata", {}).get("section", "")

    if detected_section == "Unknown":
        metadata_score = 0.5
    elif detected_section == policy_section:
        metadata_score = 1.0
    else:
        metadata_score = 0.1

    # Signal 5 — Number Parsing Confidence (NEW)
    parse_score = float(number_parsing_confidence)

    # Weighted combination
    score = (
        vector_similarity  * 0.30 +
        constraint_score   * 0.30 +
        routing_confidence * 0.20 +
        metadata_score     * 0.10 +
        parse_score        * 0.10
    )
    score = round(max(0.0, min(1.0, score)), 3)

    if score > THRESHOLD_HIGH:
        level = "high"
    elif score >= THRESHOLD_MEDIUM:
        level = "medium"
    else:
        level = "low"

    return {
        "score"  : score,
        "level"  : level,
        "components": {
            "vector_similarity"        : vector_similarity,
            "constraint_match"         : constraint_score,
            "section_routing"          : round(routing_confidence, 3),
            "metadata_strength"        : metadata_score,
            "number_parsing_confidence": round(parse_score, 3),
        }
    }

Updated enterprise_decision_v3 (full pipeline)

In [74]:
def enterprise_decision_v3(question: str, k: int = 5) -> dict:
    """
    Full Task 1 + Task 2 + Task 3 pipeline:

        question
            ↓  normalize_monetary_query    Task 3 — normalize "4k" → "4000"
            ↓  detect_ambiguity            Task 3 — flag uncertain inputs
            ↓  compute_number_parsing_confidence
            ↓  detect_section_llm          Task 1 — LLM section classifier
            ↓  hybrid_policy_retrieval     Hybrid retrieval + rule engine
            ↓  compute_confidence_v3       Task 2+3 — 5-signal confidence
            ↓  make_action                 Task 2 — escalation policy
            ↓  structured output
    """
    from datetime import datetime, timezone

    # ── Step 1: Normalize BEFORE anything else ────────────────
    normalized_query, was_normalized = normalize_monetary_query(question)

    # ── Step 2: Ambiguity detection ───────────────────────────
    ambiguity = detect_ambiguity(question, normalized_query, was_normalized)

    # ── Step 3: Number parsing confidence ─────────────────────
    num_parse_conf = compute_number_parsing_confidence(ambiguity)

    # ── Step 4: LLM section routing (on normalized query) ─────
    section_result = detect_section_llm(normalized_query)
    where_filter   = section_to_filter(section_result)

    # ── Step 5: Hybrid retrieval (on normalized query) ────────
    retrieval = hybrid_policy_retrieval(normalized_query, k=k, where=where_filter)
    best      = retrieval.get("best_match")

    # ── Step 6: No policy found → hard escalation ─────────────
    if not best:
        return {
            "question"                 : question,
            "normalized_query"         : normalized_query,
            "decision"                 : None,
            "confidence_score"         : 0.0,
            "number_parsing_confidence": num_parse_conf,
            "action"                   : "ESCALATE_TO_HUMAN",
            "reason"                   : "No matching policy found in database",
            "ambiguity"                : ambiguity,
            "timestamp"                : datetime.now(timezone.utc).isoformat(),
        }

    # ── Step 7: 5-signal confidence scoring ───────────────────
    confidence = compute_confidence_v3(
        best_match               = best,
        constraint_used          = retrieval["constraint_used"],
        section_result           = section_result,
        number_parsing_confidence= num_parse_conf,
    )

    # ── Step 8: Action + reason ───────────────────────────────
    action = make_action(confidence["level"])
    reason = build_reason(confidence, retrieval["constraint_used"], section_result)

    if ambiguity["is_ambiguous"]:
        reason = f"Ambiguous monetary input detected — {reason}"

    # ── Step 9: Decision value ────────────────────────────────
    if action == "AUTO_DECISION":
        approval       = best["metadata"].get("approval_required")
        decision_value = "AUTO_APPROVED" if approval == "No" else "REQUIRES_APPROVAL"
    else:
        decision_value = None

    return {
        "question"                 : question,
        "normalized_query"         : normalized_query,
        "was_normalized"           : was_normalized,
        "decision"                 : decision_value,
        "approval_required"        : best["metadata"].get("approval_required") if action == "AUTO_DECISION" else None,
        "risk_level"               : best["metadata"].get("risk_level"),
        "confidence_score"         : confidence["score"],
        "confidence_level"         : confidence["level"],
        "confidence_detail"        : confidence["components"],
        "number_parsing_confidence": num_parse_conf,
        "action"                   : action,
        "reason"                   : reason,
        "ambiguity"                : ambiguity,
        "constraint_used"          : retrieval["constraint_used"],
        "policy_matched"           : best["metadata"].get("policy_name"),
        "section_routing"          : {
            "detected"      : section_result["section"],
            "confidence"    : section_result["confidence"],
            "filter_applied": where_filter,
        },
        "timestamp"                : datetime.now(timezone.utc).isoformat(),
    }

Test Suite

In [75]:
# ═══════════════════════════════════════════════════════════════
# Task 3 Required Test Cases
# ═══════════════════════════════════════════════════════════════

test_cases = [
    # (query, expected_normalized_number, note)
    ("Who signs off on a 4k hardware purchase?",           4000, "k suffix"),
    ("Approval needed for $4,000 device?",                 4000, "dollar + comma"),
    ("USD 4k laptop — who approves?",                      4000, "USD prefix + k"),
    ("Need approval for 4 thousand equipment purchase?",   4000, "written-out thousand"),
    ("4k and 300 purchase — who approves?",                None, "conflicting numbers"),
]

print("=" * 70)
print("  TASK 3 — MONETARY NORMALIZATION + AMBIGUITY TEST")
print("=" * 70)

for query, expected_num, note in test_cases:

    result = enterprise_decision_v3(query)

    norm_nums  = result["ambiguity"]["normalized_numbers"]
    ambiguous  = result["ambiguity"]["is_ambiguous"]
    num_conf   = result["number_parsing_confidence"]
    conf_score = result["confidence_score"]
    action     = result["action"]

    # Check if normalization produced the expected number
    num_ok = (expected_num in norm_nums) if expected_num else True
    num_icon = "✅" if num_ok else "❌"

    action_icon = {
        "AUTO_DECISION"    : "✅",
        "ASK_CLARIFICATION": "⚠️",
        "ESCALATE_TO_HUMAN": "🚨"
    }.get(action, "❓")

    print(f"\n  [{note.upper()}]")
    print(f"  Query       : {query}")
    print(f"  Normalized  : {result['normalized_query']}")
    print(f"  Numbers     : {norm_nums}  {num_icon}  (expected {expected_num})")
    print(f"  Ambiguous   : {ambiguous}")
    print(f"  Signals     : k={result['ambiguity']['k_pattern_present']}  "
          f"small={result['ambiguity']['suspicious_small']}  "
          f"multi={result['ambiguity']['multiple_numbers']}")
    print(f"  Parse conf  : {num_conf}")
    print(f"  Total conf  : {conf_score}  ({result['confidence_level'].upper()})")
    print(f"  Action      : {action_icon}  {action}")
    print(f"  Decision    : {result['decision']}")
    print(f"  Reason      : {result['reason']}")

# ─────────────────────────────────────────────────────────────
# Score comparison: clean $4000 vs ambiguous 4k
# ─────────────────────────────────────────────────────────────
print(f"\n{'═'*70}")
print("  COMPARISON: clean input vs ambiguous input")
print(f"{'═'*70}")

clean     = enterprise_decision_v3("Purchase device with price 4000$, who approves?")
ambiguous = enterprise_decision_v3("Who signs off on a 4k hardware purchase?")

rows = [
    ("Query",         clean["question"][:45],    ambiguous["question"][:45]),
    ("Normalized to", str(clean["ambiguity"]["normalized_numbers"]),
                      str(ambiguous["ambiguity"]["normalized_numbers"])),
    ("Was normalized", str(clean["was_normalized"]),
                       str(ambiguous["was_normalized"])),
    ("Parse conf",    str(clean["number_parsing_confidence"]),
                      str(ambiguous["number_parsing_confidence"])),
    ("Constraint",    str(clean["constraint_used"]),
                      str(ambiguous["constraint_used"])),
    ("Confidence",    str(clean["confidence_score"]),
                      str(ambiguous["confidence_score"])),
    ("Action",        clean["action"],            ambiguous["action"]),
    ("Decision",      str(clean["decision"]),     str(ambiguous["decision"])),
]

print(f"\n  {'Metric':<20} {'Clean ($4000)':<30} {'Ambiguous (4k)'}")
print(f"  {'─'*65}")
for label, c_val, a_val in rows:
    print(f"  {label:<20} {c_val:<30} {a_val}")

print(f"""
{'═'*70}
  CORE PRINCIPLE DEMONSTRATED
{'═'*70}
  "$4000"  → extract_numbers sees 4000  → rule fires  → HIGH confidence  → AUTO
  "4k"     → normalize first → 4000    → rule fires  → MEDIUM confidence → ASK
             (slight confidence reduction because normalization occurred —
              system is correct but records that it had to interpret the input)

  "4k and 300" → normalize → [4000, 300] → multiple numbers →
                 parse_confidence = 0.4  → LOW confidence → ESCALATE
                 (Which number is the price? The system does not guess.)
""")

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

  TASK 3 — MONETARY NORMALIZATION + AMBIGUITY TEST
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'

  [K SUFFIX]
  Query       : Who signs off on a 4k hardware purchase?
  Normalized  : Who signs off on a 4000 hardware purchase?
  Numbers     : [4000.0]  ✅  (expected 4000)
  Ambiguous   : False
  Signals     : k=True  small=False  multi=False
  Parse conf  : 0.8
  Total conf  : 0.576  (MEDIUM)
  Action      : ⚠️  ASK_CLARIFICATION
  Decision    : None
  Reason      : moderate semantic similarity, deterministic rule matched, section unknown — no metadata filter applied
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'

  [DOLLAR + COMMA]
  Query       : Approval needed for $4,000 device?
  Normalized  : Approval needed for 4000 device?
  Numbers     : [4000.0]  ✅  (expected 4000)
  Ambiguous   : False
  Signals     : k=False  small=False  multi=False
  Parse conf  : 0.8
  Total conf  : 0.57  (MEDIUM)
  Action      :

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'
⚠️  LLM call failed: type object 'DynamicCache' has no attribute 'from_legacy_cache'

  Metric               Clean ($4000)                  Ambiguous (4k)
  ─────────────────────────────────────────────────────────────────
  Query                Purchase device with price 4000$, who approve Who signs off on a 4k hardware purchase?
  Normalized to        [4000.0]                       [4000.0]
  Was normalized       True                           True
  Parse conf           0.8                            0.8
  Constraint           True                           True
  Confidence           0.584                          0.576
  Action               ASK_CLARIFICATION              ASK_CLARIFICATION
  Decision             None                           None

══════════════════════════════════════════════════════════════════════
  CORE PRINCIPLE DEMONSTRATED
══════════════════════════════════════════════════

**How the 3 tasks connect:**
```
Raw query: "Who signs off on a 4k hardware purchase?"
        │
        ▼  Task 3 ── normalize_monetary_query()
        │             "...4k..." → "...4000..."
        │             was_normalized = True
        │
        ▼  Task 3 ── detect_ambiguity()
        │             k_pattern=True, suspicious_small=False
        │             is_ambiguous = False (normalization succeeded cleanly)
        │
        ▼  Task 3 ── compute_number_parsing_confidence()
        │             score = 0.8  (normalized but unambiguous)
        │
        ▼  Task 1 ── detect_section_llm()
        │             {"section": "Financial", "confidence": 0.88}
        │
        ▼  Hybrid ── hybrid_policy_retrieval()
        │             constraint fired → amount > 3000 ✅
        │
        ▼  Task 2+3 ─ compute_confidence_v3()
                      vector=0.6, constraint=1.0, routing=0.88,
                      metadata=1.0, parse=0.8
                      → score = 0.71 → MEDIUM → ASK_CLARIFICATION

## Task 4 — Turn the System into a ReAct Agent



### 🎯 Objective

Transform the current deterministic pipeline into a simple ReAct-style agent.

Instead of a fixed sequence of steps, the LLM must:

1. Think
2. Decide the next action
3. Call a tool
4. Observe the result
5. Produce a final answer

---

# 🏗 What You Must Build

Convert key system functions into tools:

* retrieve_policy (Chroma search)
* validate_rule (numeric constraint matching)

---

# 🧠 ReAct Format

Your agent must follow this structure internally:

```
Thought: <reason about what to do next>
Action: <tool name>
Action Input: <tool input>
Observation: <tool output>
Final Answer: <structured decision>
```

---

# 🔧 Required Tools

Example tool definitions:

```python
tools = {
    "retrieve_policy": chroma_search,
    "validate_rule": match_numeric_condition
}
```

---

# 📤 Required Final Output

The agent must return structured output:

```json
{
  "decision": "REQUIRES_APPROVAL",
  "approval_required": "Manager+Finance+CTO",
  "risk_level": "Medium",
  "reasoning": "Validated rule for amount > 3000"
}
```

---

# 🛡 Safety Constraints

You must:

* Prevent the LLM from hallucinating new rules
* Ensure validation tool confirms numeric conditions
* Fail safely if no rule matches

---

# 🧪 Testing Requirements

Test at least:

* 3 normal financial queries
* 2 HR queries
* 2 ambiguous queries
* 1 invalid input

---

# Learning Outcome

After this task, you should understand:

* Tool-based LLM systems
* ReAct reasoning pattern
* Agentic AI architecture
* Difference between pipeline and agent systems

---

🔥 These tasks are system design upgrades — not prompt writing exercises.


Tool Registry + ReAct Parser

In [76]:
# ═══════════════════════════════════════════════════════════════
# TASK 4 — ReAct Agent
#
# Architecture difference vs Tasks 1-3:
#
#   Pipeline (Tasks 1-3):
#     fixed steps → normalize → retrieve → score → decide
#     LLM only used for section classification
#
#   ReAct Agent (Task 4):
#     LLM drives the reasoning loop
#     LLM decides WHICH tool to call and WHEN to stop
#     Tools are Python functions — LLM cannot modify their logic
#     Safety: rule validation must happen via tool call, not LLM memory
#
# ReAct Loop:
#   Thought → Action → Action Input → [Python tool runs] → Observation
#   Repeat until LLM emits "Final Answer"
# ═══════════════════════════════════════════════════════════════

import json
import re
from datetime import datetime, timezone


# ─────────────────────────────────────────────────────────────
# Tool Definitions
# Each tool wraps an existing Python function with a clean
# string interface so the LLM can call it by name.
# ─────────────────────────────────────────────────────────────

def tool_retrieve_policy(query: str) -> str:
    """
    Semantic + constraint retrieval from Chroma.
    Input  : plain text query
    Output : top-3 policies as formatted string
    """
    try:
        normalized, _ = normalize_monetary_query(query)
        hits = chroma_search(normalized, k=3)

        if not hits:
            return "No policies found for this query."

        lines = []
        for i, h in enumerate(hits, 1):
            m = h["metadata"]
            lines.append(
                f"[{i}] policy_id={h['id']} | "
                f"section={m.get('section')} | "
                f"condition={m.get('condition')} | "
                f"approval={m.get('approval_required')} | "
                f"risk={m.get('risk_level')} | "
                f"distance={h['distance']:.3f}"
            )
        return "\n".join(lines)

    except Exception as e:
        return f"retrieve_policy error: {e}"


def tool_validate_rule(input_str: str) -> str:
    """
    Deterministic rule validation.
    Input  : "condition=amount > 3000, value=4000"
    Output : "MATCH" or "NO_MATCH" or "PARSE_ERROR"
    """
    try:
        # Parse  condition=..., value=...
        cond_match = re.search(r"condition\s*=\s*(.+?),\s*value\s*=\s*([\d.]+)", input_str)
        if not cond_match:
            return "PARSE_ERROR: expected format 'condition=<cond>, value=<number>'"

        condition = cond_match.group(1).strip()
        value     = float(cond_match.group(2).strip())

        result = match_numeric_condition(condition, value)
        return f"{'MATCH' if result else 'NO_MATCH'} | condition='{condition}' | value={value}"

    except Exception as e:
        return f"validate_rule error: {e}"


def tool_normalize_query(query: str) -> str:
    """
    Normalize monetary expressions in a query.
    Input  : raw query string
    Output : normalized query + flag
    """
    try:
        normalized, was_normalized = normalize_monetary_query(query)
        nums = extract_numbers(normalized)
        return (
            f"normalized='{normalized}' | "
            f"was_normalized={was_normalized} | "
            f"numbers_found={nums}"
        )
    except Exception as e:
        return f"normalize_query error: {e}"


# ─────────────────────────────────────────────────────────────
# Tool Registry
# ─────────────────────────────────────────────────────────────

TOOLS = {
    "retrieve_policy" : tool_retrieve_policy,
    "validate_rule"   : tool_validate_rule,
    "normalize_query" : tool_normalize_query,
}

TOOL_DESCRIPTIONS = """
Available tools:

  normalize_query(query)
    → Normalize monetary expressions like "4k" → 4000 before retrieval.
    → Use this FIRST when the query contains "k", "thousand", "$", commas.
    → Input: raw query string

  retrieve_policy(query)
    → Search the compliance policy database for relevant policies.
    → Returns top-3 matching policies with conditions and approvals.
    → Input: plain text query (use normalized version if available)

  validate_rule(condition=<cond>, value=<number>)
    → Deterministically check if a numeric value satisfies a policy condition.
    → ALWAYS call this before making a final decision on numeric queries.
    → Input: "condition=amount > 3000, value=4000"
    → Returns: MATCH or NO_MATCH
"""


# ─────────────────────────────────────────────────────────────
# ReAct Output Parser
# ─────────────────────────────────────────────────────────────

def parse_react_step(text: str) -> dict:
    """
    Parse one LLM output step into its ReAct components.

    Returns dict with keys:
        type         : "action" | "final_answer" | "thought_only" | "unknown"
        thought      : str
        action       : str   (tool name)
        action_input : str   (tool input)
        final_answer : str   (raw final answer text)
    """
    # Normalize whitespace
    text = text.strip()

    # Check for Final Answer first
    fa_match = re.search(
        r"Final\s+Answer\s*:\s*(.+)",
        text,
        re.IGNORECASE | re.DOTALL
    )
    if fa_match:
        return {
            "type"        : "final_answer",
            "thought"     : _extract_thought(text),
            "final_answer": fa_match.group(1).strip(),
        }

    # Check for Action + Action Input
    action_match = re.search(
        r"Action\s*:\s*(\w+)",
        text,
        re.IGNORECASE
    )
    input_match = re.search(
        r"Action\s+Input\s*:\s*(.+?)(?=\nObservation|\nThought|\nAction|\Z)",
        text,
        re.IGNORECASE | re.DOTALL
    )

    if action_match:
        return {
            "type"        : "action",
            "thought"     : _extract_thought(text),
            "action"      : action_match.group(1).strip(),
            "action_input": input_match.group(1).strip() if input_match else "",
        }

    # Thought only (LLM still reasoning, no action yet)
    thought = _extract_thought(text)
    if thought:
        return {"type": "thought_only", "thought": thought}

    return {"type": "unknown", "raw": text}


def _extract_thought(text: str) -> str:
    m = re.search(
        r"Thought\s*:\s*(.+?)(?=\nAction|\nFinal|\Z)",
        text,
        re.IGNORECASE | re.DOTALL
    )
    return m.group(1).strip() if m else ""


def parse_final_answer(raw: str) -> dict:
    """
    Extract structured JSON from the Final Answer block.
    Falls back gracefully if the LLM didn't return valid JSON.
    """
    # Try to find a JSON block
    json_match = re.search(r"\{.*?\}", raw, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            pass

    # Fallback — return raw text wrapped in a dict
    return {"raw_answer": raw, "parse_error": True}

ReAct Agent Loop

In [77]:
# ═══════════════════════════════════════════════════════════════
# ReAct Agent
# ═══════════════════════════════════════════════════════════════

MAX_STEPS = 6          # prevent infinite loops
REACT_SYSTEM_PROMPT = f"""You are a compliance decision agent. Your job is to answer
compliance policy questions by using tools — NOT by relying on memory or assumptions.

{TOOL_DESCRIPTIONS}

You must follow this EXACT format for every step:

Thought: <reason about what to do next>
Action: <tool name exactly as listed above>
Action Input: <input to the tool>

After receiving an Observation, continue with another Thought/Action/Action Input.

When you have enough information to answer, output EXACTLY:

Thought: <final reasoning>
Final Answer: {{
  "decision": "REQUIRES_APPROVAL" or "AUTO_APPROVED" or "NO_POLICY_FOUND",
  "approval_required": "<value from policy or null>",
  "risk_level": "<value from policy or null>",
  "reasoning": "<brief explanation referencing tool observations>",
  "rule_validated": true or false
}}

SAFETY RULES:
- NEVER invent or guess policy rules — always use retrieve_policy.
- NEVER make a numeric decision without calling validate_rule first.
- If no rule matches, set decision to "NO_POLICY_FOUND" and rule_validated to false.
- Output ONLY valid JSON in the Final Answer block — no extra text."""


def run_react_agent(question: str, verbose: bool = True) -> dict:
    """
    Run the ReAct agent loop for a compliance question.

    Loop:
        1. Build messages with full conversation history
        2. Call LLM
        3. Parse output → extract Action or Final Answer
        4. If Action → call Python tool → append Observation
        5. If Final Answer → parse JSON → return
        6. Repeat until Final Answer or MAX_STEPS reached
    """

    # Conversation history (system + user + assistant turns)
    messages = [
        {"role": "system",  "content": REACT_SYSTEM_PROMPT},
        {"role": "user",    "content": f"Question: {question}"},
    ]

    trace         = []    # full reasoning trace for inspection
    rule_validated = False

    if verbose:
        print(f"\n{'═'*65}")
        print(f"  AGENT QUESTION: {question}")
        print(f"{'═'*65}")

    for step in range(1, MAX_STEPS + 1):

        # ── LLM call ──────────────────────────────────────────
        try:
            output = llm_pipe(
                messages,
                max_new_tokens=300,
                do_sample=False,
                temperature=None,
                top_p=None,
                return_full_text=False,
            )
            llm_text = output[0]["generated_text"].strip()
        except Exception as e:
            return _agent_error(question, f"LLM error at step {step}: {e}", trace)

        # ── Parse step ────────────────────────────────────────
        parsed = parse_react_step(llm_text)
        trace.append({"step": step, "llm_output": llm_text, "parsed": parsed})

        if verbose:
            print(f"\n  ── Step {step} ──────────────────────────")
            if parsed.get("thought"):
                print(f"  Thought : {parsed['thought'][:120]}")

        # ── Final Answer ──────────────────────────────────────
        if parsed["type"] == "final_answer":
            structured = parse_final_answer(parsed["final_answer"])

            if verbose:
                print(f"  ✅ Final Answer received")
                print(f"  Decision    : {structured.get('decision')}")
                print(f"  Approval    : {structured.get('approval_required')}")
                print(f"  Risk        : {structured.get('risk_level')}")
                print(f"  Validated   : {structured.get('rule_validated')}")
                print(f"  Reasoning   : {structured.get('reasoning', '')[:100]}")

            return {
                "question"      : question,
                "decision"      : structured.get("decision"),
                "approval_required": structured.get("approval_required"),
                "risk_level"    : structured.get("risk_level"),
                "reasoning"     : structured.get("reasoning"),
                "rule_validated": structured.get("rule_validated", rule_validated),
                "steps_taken"   : step,
                "trace"         : trace,
                "timestamp"     : datetime.now(timezone.utc).isoformat(),
                "error"         : structured.get("parse_error"),
            }

        # ── Tool call ─────────────────────────────────────────
        if parsed["type"] == "action":
            tool_name  = parsed.get("action", "").strip()
            tool_input = parsed.get("action_input", "").strip()

            if verbose:
                print(f"  Action  : {tool_name}")
                print(f"  Input   : {tool_input[:80]}")

            # Safety — only call registered tools
            if tool_name not in TOOLS:
                observation = (
                    f"ERROR: Tool '{tool_name}' does not exist. "
                    f"Available tools: {list(TOOLS.keys())}"
                )
            else:
                observation = TOOLS[tool_name](tool_input)
                if tool_name == "validate_rule" and "MATCH" in observation:
                    rule_validated = True

            if verbose:
                print(f"  Obs     : {observation[:120]}")

            # Append assistant turn + observation to history
            messages.append({"role": "assistant", "content": llm_text})
            messages.append({
                "role"   : "user",
                "content": f"Observation: {observation}\n\nContinue."
            })

        elif parsed["type"] == "thought_only":
            # LLM still thinking — nudge it forward
            messages.append({"role": "assistant", "content": llm_text})
            messages.append({
                "role"   : "user",
                "content": "Please continue. Use a tool or provide your Final Answer."
            })

        else:
            # Unknown output — nudge
            messages.append({"role": "assistant", "content": llm_text})
            messages.append({
                "role"   : "user",
                "content": (
                    "Your output did not follow the required format. "
                    "Use: Thought / Action / Action Input  OR  Final Answer."
                )
            })

    # ── Max steps reached ─────────────────────────────────────
    if verbose:
        print(f"\n  🚨 MAX_STEPS ({MAX_STEPS}) reached without Final Answer")

    return _agent_error(
        question,
        f"Agent did not reach Final Answer within {MAX_STEPS} steps",
        trace
    )


def _agent_error(question: str, reason: str, trace: list) -> dict:
    return {
        "question"      : question,
        "decision"      : None,
        "approval_required": None,
        "risk_level"    : None,
        "reasoning"     : reason,
        "rule_validated": False,
        "steps_taken"   : len(trace),
        "trace"         : trace,
        "timestamp"     : datetime.now(timezone.utc).isoformat(),
        "error"         : True,
    }

 Test Suite + Pipeline vs Agent Comparison

In [78]:
# ═══════════════════════════════════════════════════════════════
# Test Suite
# ═══════════════════════════════════════════════════════════════

test_cases = {

    "FINANCIAL": [
        "Purchase device with price 4000$, who approves?",
        "Do I need approval to buy a $1200 laptop?",
        "Who signs off on a $5000 server purchase?",
    ],

    "HR": [
        "How many sick days do employees get per year?",
        "What is the process for requesting parental leave?",
    ],

    "AMBIGUOUS": [
        "Who signs off on a 4k hardware purchase?",   # "4k" — needs normalization
        "Can I share data with a vendor?",            # cross-section ambiguity
    ],

    "INVALID": [
        "asdkjhaksjdhaksjdh random nonsense query",   # no valid policy
    ],
}

# ─────────────────────────────────────────────────────────────
# Run all test cases
# ─────────────────────────────────────────────────────────────
all_results = []

for category, queries in test_cases.items():
    print(f"\n{'═'*65}")
    print(f"  CATEGORY: {category}")
    print(f"{'═'*65}")

    for q in queries:
        result = run_react_agent(q, verbose=True)
        result["category"] = category
        all_results.append(result)

# ─────────────────────────────────────────────────────────────
# Summary table
# ─────────────────────────────────────────────────────────────
print(f"\n\n{'═'*75}")
print("  AGENT RESULTS SUMMARY")
print(f"{'═'*75}")
print(f"  {'Cat':<12} {'Decision':<20} {'Approval':<22} {'Valid':<6} {'Steps':<6} {'Query'}")
print(f"  {'─'*73}")

for r in all_results:
    valid_icon = "✅" if r["rule_validated"] else "⚠️"
    error_icon = "🚨" if r.get("error") else ""
    print(
        f"  {r['category']:<12} "
        f"{str(r['decision']):<20} "
        f"{str(r['approval_required']):<22} "
        f"{valid_icon:<6} "
        f"{r['steps_taken']:<6} "
        f"{r['question'][:35]}{error_icon}"
    )


# ─────────────────────────────────────────────────────────────
# Pipeline vs Agent comparison
# ─────────────────────────────────────────────────────────────
print(f"\n\n{'═'*65}")
print("  PIPELINE vs REACT AGENT — ARCHITECTURE COMPARISON")
print(f"{'═'*65}")

comparison_query = "Purchase device with price 4000$, who approves?"

import time

# Pipeline (Task 1+2+3)
t0       = time.time()
pipeline_result = enterprise_decision_v3(comparison_query)
pipeline_ms     = round((time.time() - t0) * 1000, 1)

# ReAct Agent
t0       = time.time()
agent_result    = run_react_agent(comparison_query, verbose=False)
agent_ms        = round((time.time() - t0) * 1000, 1)

rows = [
    ("Query",           comparison_query[:45],              comparison_query[:45]),
    ("Decision",        str(pipeline_result["decision"]),   str(agent_result["decision"])),
    ("Approval",        str(pipeline_result["approval_required"]),
                        str(agent_result["approval_required"])),
    ("Rule validated",  str(pipeline_result["constraint_used"]),
                        str(agent_result["rule_validated"])),
    ("Confidence",      str(pipeline_result["confidence_score"]),   "N/A (agent decides)"),
    ("Steps",           "fixed (6 steps always)",           f"{agent_result['steps_taken']} steps (dynamic)"),
    ("Latency",         f"{pipeline_ms} ms",                f"{agent_ms} ms"),
    ("LLM role",        "Section classifier only",          "Drives full reasoning loop"),
    ("Tool control",    "Python controls sequence",         "LLM chooses tool order"),
    ("Hallucination risk", "Low (LLM sandboxed)",           "Low (tools are Python functions)"),
]

print(f"\n  {'Dimension':<22} {'Pipeline (Tasks 1-3)':<30} {'ReAct Agent (Task 4)'}")
print(f"  {'─'*73}")
for label, p_val, a_val in rows:
    print(f"  {label:<22} {p_val:<30} {a_val}")

print(f"""
{'═'*65}
  KEY INSIGHT
{'═'*65}

  Pipeline  → LLM is a small part (classifier)
              Sequence is hardcoded in Python
              Predictable, fast, easy to test

  ReAct Agent → LLM drives the entire reasoning loop
                Sequence is dynamic — agent decides tool order
                Can handle novel question structures the pipeline wasn't designed for
                Slower (multiple LLM calls), harder to predict

  Both architectures share the SAME safety guarantee:
  Rule validation is always a Python function.
  The LLM observes the result — it never modifies the rule logic.
""")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


═════════════════════════════════════════════════════════════════
  CATEGORY: FINANCIAL
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  AGENT QUESTION: Purchase device with price 4000$, who approves?
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  AGENT QUESTION: Do I need approval to buy a $1200 laptop?
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  AGENT QUESTION: Who signs off on a $5000 server purchase?
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  CATEGORY: HR
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  AGENT QUESTION: How many sick days do emplo

```

---

**How the ReAct loop works for `"Purchase device with price 4000$"`:**
```
User:  "Purchase device with price 4000$, who approves?"
         │
Step 1   Thought: I need to find relevant policies. First normalize query.
         Action: normalize_query
         Action Input: Purchase device with price 4000$
         Observation: normalized='Purchase device with price 4000'
                      numbers_found=[4000.0]
         │
Step 2   Thought: Query is clean. Now retrieve matching policies.
         Action: retrieve_policy
         Action Input: Purchase device with price 4000
         Observation: [1] condition=amount > 3000 | approval=Manager+Finance+CTO
                      [2] condition=1500 < amount <= 3000 | approval=Manager
                      [3] condition=amount <= 1500 | approval=No
         │
Step 3   Thought: Policy 1 looks right for 4000. Must validate before deciding.
         Action: validate_rule
         Action Input: condition=amount > 3000, value=4000
         Observation: MATCH | condition='amount > 3000' | value=4000.0
         │
Step 4   Thought: Rule confirmed. I have enough to answer.
         Final Answer: {
           "decision": "REQUIRES_APPROVAL",
           "approval_required": "Manager+Finance+CTO",
           "risk_level": "Medium",
           "reasoning": "validate_rule confirmed amount > 3000 for value 4000",
           "rule_validated": true
         }